# Singapore HDB Resale Prices: When Does a Town Premium Disappear?

This notebook investigates one question:

> **How much of the observed price difference between HDB towns reflects location, and how much reflects the mix of flats sold in each town?**

The analysis follows 222,066 resale registrations from January 2017 through December 2025. It first establishes data quality and representation, then follows Town coefficients through a sequence of hedonic regressions. The central non-obvious finding is a sign reversal for PASIR RIS relative to BEDOK. A final chronological prediction experiment tests whether the same preparation decisions improve performance on 2025 transactions.

The source records resale **registration dates** and excludes some transaction types, including sales between relatives and partial-share sales. Floor area can include purchased recess space, upgrading additions, or roof terraces. Results therefore describe the published transaction sample rather than all HDB homes.

Source: [HDB / data.gov.sg](https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view).

**AI declaration:** OpenAI Codex assisted with code, analytical suggestions, and writing. The work was checked by executing the complete notebook, inspecting outputs, and verifying the chronological encodings and model comparisons. Group members remain responsible for understanding and independently verifying the submitted analysis.

## 1. Research design

The tempting interpretation of a Town price map is that high-price Towns command a location premium. That interpretation can be misleading because Towns sell different mixtures of floor areas, flat types, models, storeys, remaining leases, and calendar periods.

The explanatory analysis therefore estimates the same Town coefficients after adding controls in a fixed sequence:

| Model | Controls added |
|---|---|
| M0 | Town only |
| M1 | Registration month |
| M2 | Log floor area |
| M3 | Flat Type and Flat Model |
| M4 | Storey Range and remaining lease |

The main conclusion concerns how the Town coefficients change. M4 remains an observational hedonic regression: it estimates conditional differences, not the causal effect of moving the same flat between Towns.

The prediction analysis then compares minimally processed and systematically prepared data using the same basic method, OLS, and the same 2025 test records. Accuracy is evidence about preparation choices rather than the research goal.

## 2. Reproducible data snapshot

Run `uv sync --locked`, followed by `uv run jupyter lab`, from the project root. The notebook normally reuses the local snapshot. Set `REFRESH_DATA = True` only to replace it with a new official download, still filtered through December 2025. Metadata records the source, retrieval time, cutoff, and SHA-256 checksum.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import io
import json
import time

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, filename):
    """Save a presentation-ready copy of a figure."""
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return path

DATASET_ID = "d_8b84c4ee58e3cfc0ece0d773c8ca6abc"
SOURCE_URL = f"https://data.gov.sg/datasets/{DATASET_ID}/view"
CUTOFF = "2025-12"
REFRESH_DATA = False
SNAPSHOT = Path("data/raw/hdb_resale_2017_2025.csv")
METADATA = SNAPSHOT.with_suffix(".metadata.json")

In [2]:
if REFRESH_DATA or not SNAPSHOT.exists():
    api = f"https://api-open.data.gov.sg/v1/public/api/datasets/{DATASET_ID}"
    response = requests.get(f"{api}/initiate-download", timeout=60)
    response.raise_for_status()
    download = requests.get(f"{api}/poll-download", timeout=60).json()["data"]
    while not download.get("url"):
        time.sleep(15)
        download = requests.get(f"{api}/poll-download", timeout=60).json()["data"]

    response = requests.get(download["url"], timeout=120)
    response.raise_for_status()
    source = pd.read_csv(io.BytesIO(response.content), dtype="string")
    source_month = pd.to_datetime(source["month"], format="%Y-%m")
    snapshot = source.loc[source_month.between("2017-01-01", "2025-12-31")].copy()
    SNAPSHOT.parent.mkdir(parents=True, exist_ok=True)
    snapshot.to_csv(SNAPSHOT, index=False)
    metadata = {
        "dataset_id": DATASET_ID,
        "source_url": SOURCE_URL,
        "retrieved_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
        "cutoff_month": CUTOFF,
        "source_rows": len(source),
        "source_latest_month": source["month"].max(),
        "snapshot_rows": len(snapshot),
        "snapshot_first_month": snapshot["month"].min(),
        "snapshot_last_month": snapshot["month"].max(),
        "snapshot_sha256": hashlib.sha256(SNAPSHOT.read_bytes()).hexdigest(),
    }
    METADATA.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

metadata = json.loads(METADATA.read_text(encoding="utf-8"))
assert hashlib.sha256(SNAPSHOT.read_bytes()).hexdigest() == metadata["snapshot_sha256"]
display(pd.Series(metadata, name="Snapshot metadata").to_frame())

,Snapshot metadata
dataset_id,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
source_url,https://data.gov.sg/datasets/d_8b84c...
retrieved_at,2026-10-11T21:23:29.895039+08:00
cutoff_month,2025-12
source_rows,242454
source_latest_month,2026-10
snapshot_rows,222066
snapshot_first_month,2017-01
snapshot_last_month,2025-12
snapshot_sha256,af2ed6574d542771426bf2236c122470d64f...


## 3. Raw profile and literal data quality

Use pandas' native inspection methods before conversion. `block` is read as a string because labels such as `273A` are identifiers. The literal audit checks missing values, blanks, outer whitespace, numeric parsing, label variants, and repeated public records.

The snapshot contains **308 groups of identical rows: 307 pairs and one triplet**. These groups contain 617 rows in total and create 309 excess copies beyond one retained representative per group. Because the records have no transaction ID, exact duplicates cannot be distinguished from separate sales with identical public attributes. For a conservative modelling sample, the cleaning rule keeps the first record in each group and excludes the 309 excess copies. The same rule is applied before all later EDA, explanatory regressions, and prediction models.


In [3]:
raw = pd.read_csv(SNAPSHOT, dtype={"block": "string"})
display(raw.head())
raw.info()
display(raw.dtypes.rename("Inferred dtype").to_frame())
display(raw.describe())
display(raw.describe(include=["object", "string"]))

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.000,Improved,1979,61 years 04 months,232000.000
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.000,New Generation,1978,60 years 07 months,250000.000
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.000,New Generation,1980,62 years 05 months,262000.000
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.000,New Generation,1980,62 years 01 month,265000.000
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.000,New Generation,1980,62 years 05 months,265000.000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222066 entries, 0 to 222065
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                222066 non-null  object 
 1   town                 222066 non-null  object 
 2   flat_type            222066 non-null  object 
 3   block                222066 non-null  string 
 4   street_name          222066 non-null  object 
 5   storey_range         222066 non-null  object 
 6   floor_area_sqm       222066 non-null  float64
 7   flat_model           222066 non-null  object 
 8   lease_commence_date  222066 non-null  int64  
 9   remaining_lease      222066 non-null  object 
 10  resale_price         222066 non-null  float64
dtypes: float64(2), int64(1), object(7), string(1)
memory usage: 18.6+ MB


,Inferred dtype
month,object
town,object
flat_type,object
block,string[python]
street_name,object
storey_range,object
floor_area_sqm,float64
flat_model,object
lease_commence_date,int64
remaining_lease,object


,floor_area_sqm,lease_commence_date,resale_price
count,222066.000,222066.000,222066.000
mean,96.770,1996.435,524464.323
std,24.020,14.301,186593.819
min,31.000,1966.000,140000.000
25%,81.000,1985.000,385000.000
50%,93.000,1997.000,492000.000
75%,112.000,2012.000,630000.000
max,366.700,2021.000,1658888.000


,month,town,flat_type,block,street_name,storey_range,flat_model,remaining_lease
count,222066,222066,222066,222066,222066,222066,222066,222066
unique,108,26,7,2745,576,17,21,694
top,2024-07,SENGKANG,4 ROOM,2,YISHUN RING RD,04 TO 06,Model A,94 years 10 months
freq,3036,18155,94091,669,3178,50947,79348,1913


In [4]:
literal = pd.read_csv(SNAPSHOT, dtype="string")
quality = pd.DataFrame({
    "missing_count": literal.isna().sum(),
    "blank_count": literal.apply(lambda s: s.str.strip().eq("").sum()),
    "outer_whitespace_count": literal.apply(lambda s: s.ne(s.str.strip()).sum()),
    "unique_count": literal.nunique(),
})
display(quality)

duplicate_counts = raw.groupby(list(raw.columns), dropna=False).size().rename("occurrences")
duplicate_groups = duplicate_counts.loc[duplicate_counts.gt(1)].reset_index()
duplicate_groups.insert(0, "duplicate_group", np.arange(1, len(duplicate_groups) + 1))
with pd.option_context("display.max_columns", None):
    example_rows = raw.reset_index(names="source_row").merge(
        duplicate_groups.head(3), on=list(raw.columns)
    ).sort_values(["duplicate_group", "source_row"])
    display(example_rows[["duplicate_group", "occurrences", "source_row"] + list(raw.columns)])

,missing_count,blank_count,outer_whitespace_count,unique_count
month,0,0,0,108
town,0,0,0,26
flat_type,0,0,0,7
block,0,0,0,2745
street_name,0,0,0,576
storey_range,0,0,0,17
floor_area_sqm,0,0,0,343
flat_model,0,0,0,21
lease_commence_date,0,0,0,56
remaining_lease,0,0,0,694


,duplicate_group,occurrences,source_row,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,1,2,224,2017-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81.000,Improved,1975,57 years,470000.000
1,1,2,243,2017-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81.000,Improved,1975,57 years,470000.000
2,2,2,304,2017-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68.000,Improved,1979,61 years 02 months,470000.000
3,2,2,305,2017-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68.000,Improved,1979,61 years 02 months,470000.000
4,3,2,505,2017-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101.000,Model A,1998,80 years 09 months,470000.000
5,3,2,510,2017-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101.000,Model A,1998,80 years 09 months,470000.000


In [5]:
text_columns = ["town", "flat_type", "block", "street_name", "storey_range", "flat_model"]
variant_tables = []
for column in text_columns:
    labels = literal[column].dropna().drop_duplicates().to_frame("label")
    labels["normalized_label"] = labels["label"].str.strip().str.casefold()
    labels["column"] = column
    variant_tables.append(labels.loc[labels.duplicated("normalized_label", keep=False)])
display(pd.concat(variant_tables, ignore_index=True))

numeric_source_columns = ["floor_area_sqm", "resale_price", "lease_commence_date"]
display(pd.DataFrame({
    "parse_failure_count": {
        c: int((literal[c].notna() & pd.to_numeric(literal[c], errors="coerce").isna()).sum())
        for c in numeric_source_columns
    }
}))

,label,normalized_label,column


,parse_failure_count
floor_area_sqm,0
resale_price,0
lease_commence_date,0


## 4. Type conversion and cleaning decision

Convert Month to a date and elapsed-month index, Remaining Lease to fractional years, and Storey Range to ordered intervals. Storey Range identifies the interval in which the flat is located; it does not describe the number of floors occupied.

All records pass the basic missing, parsing, positivity, date, storey, and lease-domain rules. A separate consistency check compares recorded remaining lease with `99 - (registration year - lease start year)`. The 128 records differing by more than one year are excluded because their cause cannot be resolved from the dataset. Exact duplicate excess rows are then removed by keeping one representative per identical group. `lease_commence_date` is omitted from models because registration time and remaining lease already carry approximately the same information.


In [6]:
df = raw.copy()
for column in literal.columns.difference(numeric_source_columns):
    df[column] = literal[column].str.strip().replace("", pd.NA)
for column in ["town", "flat_type", "street_name"]:
    df[column] = df[column].str.upper()
for column in numeric_source_columns:
    df[column] = pd.to_numeric(literal[column], errors="coerce")

df["transaction_date"] = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
df["transaction_year"] = df["transaction_date"].dt.year
df["month_index"] = (df["transaction_year"] - 2017) * 12 + df["transaction_date"].dt.month - 1

storeys = df["storey_range"].str.extract(r"^(\d{2}) TO (\d{2})$").astype(float)
df["storey_lower"] = storeys[0]
df["storey_upper"] = storeys[1]
storey_order = (
    df[["storey_range", "storey_lower"]].dropna().drop_duplicates()
    .sort_values("storey_lower")["storey_range"].tolist()
)
storey_mapping = {label: rank for rank, label in enumerate(storey_order)}
df["storey_ordinal"] = df["storey_range"].map(storey_mapping)

lease_parts = df["remaining_lease"].str.extract(r"^(\d+) years?(?: (\d+) months?)?$").astype(float)
df["remaining_lease_years"] = lease_parts[0] + lease_parts[1].fillna(0) / 12

In [7]:
issues = pd.DataFrame(index=df.index)
issues["missing_required_field"] = df[raw.columns].isna().any(axis=1)
issues["invalid_month"] = df["transaction_date"].isna() | ~df["month"].str.fullmatch(r"\d{4}-\d{2}", na=False)
issues["outside_snapshot_period"] = ~df["transaction_date"].between("2017-01-01", "2025-12-31")
for column in numeric_source_columns:
    issues[f"invalid_{column}"] = ~np.isfinite(df[column]) | df[column].le(0)
issues["invalid_lease_start_year"] = (
    df["lease_commence_date"].mod(1).ne(0)
    | df["lease_commence_date"].gt(df["transaction_year"])
)
issues["invalid_storey_range"] = (
    df["storey_lower"].isna() | df["storey_lower"].le(0)
    | df["storey_upper"].lt(df["storey_lower"])
)
issues["invalid_remaining_lease"] = (
    df["remaining_lease_years"].isna()
    | ~df["remaining_lease_years"].between(0, 99)
    | lease_parts[1].fillna(0).ge(12)
)
display(issues.sum().rename("Flagged rows").to_frame())

invalid_rows = issues.any(axis=1)
invalid_records = pd.concat([df.loc[invalid_rows], issues.loc[invalid_rows]], axis=1)
display(invalid_records.head(10))
df = df.loc[~invalid_rows].copy()
display(pd.Series({
    "snapshot_rows": len(raw),
    "removed_basic_invalid_rows": int(invalid_rows.sum()),
    "rows_after_basic_checks": len(df),
}, name="Basic validation decision").to_frame())


,Flagged rows
missing_required_field,0
invalid_month,0
outside_snapshot_period,0
invalid_floor_area_sqm,0
invalid_resale_price,0
invalid_lease_commence_date,0
invalid_lease_start_year,0
invalid_storey_range,0
invalid_remaining_lease,0


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,...,outside_snapshot_period,invalid_floor_area_sqm,invalid_resale_price,invalid_lease_commence_date,invalid_lease_start_year,invalid_storey_range,invalid_remaining_lease


,Basic validation decision
snapshot_rows,222066
removed_basic_invalid_rows,0
rows_after_basic_checks,222066


In [8]:
expected_lease = 99 - (df["transaction_year"] - df["lease_commence_date"])
lease_residual = df["remaining_lease_years"] - expected_lease
lease_exclusion = lease_residual.abs().gt(1 + 1e-8)
lease_excluded = df.loc[lease_exclusion].copy()
lease_excluded["lease_residual_years"] = lease_residual.loc[lease_exclusion]
df = df.loc[~lease_exclusion].copy()

duplicate_exclusion = df.duplicated(subset=list(raw.columns), keep="first")
duplicate_excluded = df.loc[duplicate_exclusion].copy()
df = df.loc[~duplicate_exclusion].copy()

assert not df.duplicated(subset=list(raw.columns)).any()
assert df.index.intersection(lease_excluded.index).empty
assert df.index.intersection(duplicate_excluded.index).empty
display(pd.Series({
    "lease_inconsistent_rows_excluded": len(lease_excluded),
    "duplicate_excess_rows_excluded": len(duplicate_excluded),
    "retained_analysis_rows": len(df),
    "retained_pct": 100 * len(df) / len(raw),
}, name="Final cleaning decision").to_frame())


,Final cleaning decision
lease_inconsistent_rows_excluded,128.000
duplicate_excess_rows_excluded,309.000
retained_analysis_rows,221629.000
retained_pct,99.803


## 5. Feature preparation

Flat Type and Flat Model are categorical. Their full one-hot encodings are retained in data tables, while pie charts combine individually sub-1% slices only for readability. Storey Range receives an ordinal code for ordered summaries, but the explanatory regression uses categorical intervals so that it does not impose equal price increments between ranges.

Address must be treated hierarchically. `block` alone is not a location; the building-level public address is Town + Street Name + Block.

In [9]:
def category_pie(shares, ax, title):
    """Combine small slices for display while preserving the original statistics."""
    visible = shares.loc[shares.ge(1)].copy()
    small = shares.loc[shares.lt(1)].sum()
    if small > 0:
        visible.loc["Other (<1% each)"] = small
    wedges, _, _ = ax.pie(
        visible, startangle=90,
        autopct=lambda pct: f"{pct:.1f}%" if pct >= 5 else "",
        colors=sns.color_palette("tab20", len(visible)),
        wedgeprops={"edgecolor": "white"},
    )
    labels = [f"{label}: {share:.2f}%" for label, share in visible.items()]
    ax.legend(wedges, labels, loc="center left", bbox_to_anchor=(1, .5), fontsize=9)
    ax.set_title(title)


category_columns = ["flat_type", "flat_model"]
category_tables = {}
for column in category_columns:
    counts = df[column].value_counts()
    table = counts.rename("count").to_frame()
    table["share_pct"] = counts / len(df) * 100
    table["rare_below_1pct"] = table["share_pct"].lt(1)
    category_tables[column] = table
    print(f"Category distribution: {column}")
    display(table)

one_hot = pd.get_dummies(df[category_columns], prefix=category_columns, dtype=int)
display(one_hot.head())
display(one_hot.mean().mul(100).rename("One-hot share (%)").to_frame())
for column in category_columns:
    assert one_hot.filter(like=f"{column}_").sum(axis=1).eq(1).all()

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for column, ax in zip(category_columns, axes):
    category_pie(category_tables[column]["share_pct"], ax, f"{column}: transaction shares")
plt.tight_layout()
save_figure(fig, "01_category_shares.png")
plt.show()

Category distribution: flat_type


,count,share_pct,rare_below_1pct
flat_type,,,
4 ROOM,93853,42.347,False
5 ROOM,54416,24.553,False
3 ROOM,52785,23.817,False
EXECUTIVE,15926,7.186,False
2 ROOM,4486,2.024,False
MULTI-GENERATION,84,0.038,True
1 ROOM,79,0.036,True


Category distribution: flat_model


,count,share_pct,rare_below_1pct
flat_model,,,
Model A,79163,35.719,False
Improved,54176,24.444,False
New Generation,27144,12.247,False
Premium Apartment,24441,11.028,False
Simplified,8494,3.833,False
Apartment,7976,3.599,False
Maisonette,6113,2.758,False
Standard,5953,2.686,False
DBSS,3553,1.603,False


,flat_type_1 ROOM,flat_type_2 ROOM,flat_type_3 ROOM,flat_type_4 ROOM,flat_type_5 ROOM,flat_type_EXECUTIVE,flat_type_MULTI-GENERATION,...,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0
4,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0


,One-hot share (%)
flat_type_1 ROOM,0.036
flat_type_2 ROOM,2.024
flat_type_3 ROOM,23.817
flat_type_4 ROOM,42.347
flat_type_5 ROOM,24.553
flat_type_EXECUTIVE,7.186
flat_type_MULTI-GENERATION,0.038
flat_model_2-room,0.163
flat_model_3Gen,0.030
flat_model_Adjoined flat,0.171


/tmp/ipykernel_61370/3130341391.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
storey_table = df.groupby("storey_range").agg(
    ordinal=("storey_ordinal", "first"),
    lower=("storey_lower", "first"),
    upper=("storey_upper", "first"),
    count=("resale_price", "size"),
).reindex(storey_order)
storey_table["share_pct"] = storey_table["count"] / len(df) * 100
display(storey_table)
fig, ax = plt.subplots(figsize=(9, 5))
category_pie(storey_table["share_pct"], ax, "Storey range: transaction shares")
plt.tight_layout()
save_figure(fig, "02_storey_shares.png")
plt.show()

,ordinal,lower,upper,count,share_pct
storey_range,,,,,
01 TO 03,0,1.000,3.000,39086,17.636
04 TO 06,1,4.000,6.000,50849,22.943
07 TO 09,2,7.000,9.000,46482,20.973
10 TO 12,3,10.000,12.000,41399,18.679
13 TO 15,4,13.000,15.000,21333,9.626
16 TO 18,5,16.000,18.000,9979,4.503
19 TO 21,6,19.000,21.000,4295,1.938
22 TO 24,7,22.000,24.000,3014,1.360
25 TO 27,8,25.000,27.000,1872,0.845


/tmp/ipykernel_61370/2086283453.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
address_levels = [
    ["town"], ["street_name"], ["block"],
    ["town", "street_name"], ["town", "block"], ["street_name", "block"],
    ["town", "street_name", "block"],
]
address_summary = []
for columns in address_levels:
    sizes = df.groupby(columns).size()
    address_summary.append({
        "level": " + ".join(columns),
        "unique_combinations": len(sizes),
        "min_transactions": sizes.min(),
        "median_transactions": sizes.median(),
        "max_transactions": sizes.max(),
    })
address_summary = pd.DataFrame(address_summary)
display(address_summary)

address_counts = df.groupby(["town", "street_name", "block"]).size().rename("transactions")
display(address_counts.sort_values(ascending=False).head(10).to_frame())
display(pd.Series({
    "street_names_in_multiple_towns": int(df.groupby("street_name")["town"].nunique().gt(1).sum()),
    "block_labels_on_multiple_streets": int(df.groupby("block")["street_name"].nunique().gt(1).sum()),
}, name="Address ambiguity").to_frame())

,level,unique_combinations,min_transactions,median_transactions,max_transactions
0,town,26,542,7565.000,18108
1,street_name,576,1,277.500,3175
2,block,2745,1,51.000,667
3,town + street_name,576,1,277.500,3175
4,town + block,9097,1,21.000,177
5,street_name + block,9670,1,20.000,171
6,town + street_name + block,9670,1,20.000,171


transactions
town        street_name      block              
PUNGGOL     PUNGGOL WALK     308A            171
                             308C            154
QUEENSTOWN  DAWSON RD        87              140
PUNGGOL     PUNGGOL WALK     310B            139
TOA PAYOH   LOR 1A TOA PAYOH 139A            139
QUEENSTOWN  DAWSON RD        88              134
CLEMENTI    CLEMENTI AVE 4   311B            124
JURONG WEST BOON LAY AVE     187             123
QUEENSTOWN  DAWSON RD        86              123
PUNGGOL     PUNGGOL WALK     308B            116

,Address ambiguity
street_names_in_multiple_towns,0
block_labels_on_multiple_streets,1499


### 5.1 Geographic proxy and historical address encoding

The official URA Master Plan 2019 planning-area polygons provide an aggregate map. They are proxies rather than verified HDB-town or building boundaries. `KALLANG/WHAMPOA` is associated with Kallang, and the eleven Central Area planning polygons are dissolved to `CENTRAL AREA`.

For prediction, the address hierarchy is encoded as global → Town → Town + Street → Town + Street + Block. Each level shrinks towards its parent. Every year's encoding uses only earlier years, so a transaction's own price and future prices never enter its feature. The address encoding is used for prediction only; it is not interpreted as a causal location coefficient.

In [12]:
BOUNDARY_ID = "d_4765db0e87b9c86336792efe8a1f7a66"
BOUNDARY_URL = f"https://data.gov.sg/datasets/{BOUNDARY_ID}/view"
BOUNDARY_PATH = Path("data/raw/ura_mp2019_planning_areas.geojson")
if not BOUNDARY_PATH.exists():
    url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{BOUNDARY_ID}/poll-download"
    download_url = requests.get(url, timeout=60).json()["data"]["url"]
    response = requests.get(download_url, timeout=120)
    response.raise_for_status()
    BOUNDARY_PATH.write_bytes(response.content)
    boundary_metadata = {
        "dataset_id": BOUNDARY_ID, "source_url": BOUNDARY_URL,
        "retrieved_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
        "sha256": hashlib.sha256(response.content).hexdigest(),
    }
    BOUNDARY_PATH.with_suffix(".metadata.json").write_text(
        json.dumps(boundary_metadata, indent=2) + "\n", encoding="utf-8"
    )

areas = gpd.read_file(BOUNDARY_PATH).to_crs(epsg=3414)
areas["map_town"] = areas["PLN_AREA_N"].replace({"KALLANG": "KALLANG/WHAMPOA"})
areas.loc[areas["CA_IND"].eq("Y"), "map_town"] = "CENTRAL AREA"
town_boundaries = areas[["map_town", "geometry"]].dissolve(by="map_town").reset_index()
town_statistics = df.groupby("town").agg(
    transactions=("resale_price", "size"), median_price=("resale_price", "median")
)
town_statistics["share_pct"] = town_statistics["transactions"] / len(df) * 100
town_map = town_boundaries.merge(town_statistics, left_on="map_town", right_index=True, how="left")
display(pd.Series({
    "HDB_town_labels": df["town"].nunique(),
    "matched_town_labels": df["town"].isin(town_boundaries["map_town"]).groupby(df["town"]).all().sum(),
    "Central_Area_planning_polygons": areas["CA_IND"].eq("Y").sum(),
}, name="Map matching audit").to_frame())
display(town_statistics.sort_values("transactions", ascending=False).head())

,Map matching audit
HDB_town_labels,26
matched_town_labels,26
Central_Area_planning_polygons,11


,transactions,median_price,share_pct
town,,,
SENGKANG,18108,515000.000,8.170
PUNGGOL,16105,527000.000,7.267
WOODLANDS,15726,455000.000,7.096
TAMPINES,15127,538888.000,6.825
YISHUN,15113,428000.000,6.819


In [13]:
address_columns = ["town", "street_name", "block"]


def hierarchical_target_encode(reference, addresses):
    """Fit hierarchical means on reference prices and transform address-only rows."""
    global_mean = reference["resale_price"].mean()
    result = pd.DataFrame(index=addresses.index)
    parent = None
    for level, (strength, name) in enumerate(zip([50, 20, 10], ["town_te", "street_te", "address_te"]), 1):
        keys = address_columns[:level]
        stats = reference.groupby(keys)["resale_price"].agg(["sum", "count"]).reset_index()
        if level == 1:
            stats["prior"] = global_mean
            fallback = global_mean
        else:
            stats = stats.merge(parent.rename(columns={"encoding": "prior"}), on=keys[:-1])
            fallback = result.iloc[:, -1]
        stats["encoding"] = (stats["sum"] + strength * stats["prior"]) / (stats["count"] + strength)
        parent = stats[keys + ["encoding"]]
        mapped = addresses[keys].merge(parent, on=keys, how="left", sort=False)["encoding"]
        result[name] = pd.Series(mapped.to_numpy(), index=addresses.index).fillna(fallback)
    return result


encoding_folds = []
for year in sorted(df["transaction_year"].unique()):
    if year == 2017:
        continue
    reference = df.loc[df["transaction_year"].lt(year)]
    current = df.loc[df["transaction_year"].eq(year)]
    fold = hierarchical_target_encode(reference, current[address_columns])
    fold["encoding_year"] = int(year)
    fold["reference_last_year"] = int(reference["transaction_year"].max())
    fold["split"] = "test" if year == 2025 else "train"
    encoding_folds.append(fold)

address_encoding = pd.concat(encoding_folds).sort_index()
df["address_te"] = address_encoding["address_te"]
assert address_encoding["reference_last_year"].lt(address_encoding["encoding_year"]).all()
assert address_encoding[["town_te", "street_te", "address_te"]].notna().all().all()
display(address_encoding.groupby(["encoding_year", "reference_last_year", "split"]).size().rename("rows").to_frame())
display(df.loc[address_encoding.index, ["month"] + address_columns].join(
    address_encoding[["town_te", "street_te", "address_te"]]
).head(8))

,,,rows
encoding_year,reference_last_year,split,
2018,2017,train,21548
2019,2018,train,22167
2020,2019,train,23313
2021,2020,train,29051
2022,2021,train,26697
2023,2022,train,25710
2024,2023,train,27797
2025,2024,test,25009


,month,town,street_name,block,town_te,street_te,address_te
20509,2018-01,ANG MO KIO,ANG MO KIO AVE 3,314,423179.754,380353.593,321325.643
20510,2018-01,ANG MO KIO,ANG MO KIO AVE 4,109,423179.754,380872.973,357219.396
20511,2018-01,ANG MO KIO,ANG MO KIO AVE 5,150,423179.754,386220.700,405939.000
20512,2018-01,ANG MO KIO,ANG MO KIO AVE 10,559,423179.754,360228.816,343819.211
20513,2018-01,ANG MO KIO,ANG MO KIO AVE 10,461,423179.754,360228.816,348806.297
20514,2018-01,ANG MO KIO,ANG MO KIO AVE 10,466,423179.754,360228.816,338877.726
20515,2018-01,ANG MO KIO,ANG MO KIO AVE 5,152,423179.754,386220.700,386220.700
20516,2018-01,ANG MO KIO,ANG MO KIO AVE 3,348,423179.754,380353.593,387627.994


## 6. Numeric distributions and valid tails

Summarize area, price, remaining lease, storey, time, and lease start year. Histograms combine a KDE with quartile lines. Storey uses a wider smoothing bandwidth because its code is discrete.

IQR flags describe distribution tails rather than confirmed errors. Area is also checked within Flat Type because a globally unusual area can be ordinary for a larger flat. Price and area tails remain in the main sample unless a model comparison provides evidence for removal.

In [14]:
numeric_columns = [
    "floor_area_sqm", "resale_price", "remaining_lease_years",
    "storey_ordinal", "month_index", "lease_commence_date",
]
numeric_summary = df[numeric_columns].describe(percentiles=[.01, .25, .5, .75, .99]).T
numeric_summary["skewness"] = df[numeric_columns].skew()
display(numeric_summary)

q1 = df[numeric_columns].quantile(.25)
q3 = df[numeric_columns].quantile(.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outlier_flags = df[numeric_columns].lt(lower_fence) | df[numeric_columns].gt(upper_fence)
outlier_summary = pd.DataFrame({
    "lower_fence": lower_fence, "upper_fence": upper_fence,
    "lower_tail_count": df[numeric_columns].lt(lower_fence).sum(),
    "upper_tail_count": df[numeric_columns].gt(upper_fence).sum(),
    "outlier_count": outlier_flags.sum(),
    "outlier_pct": outlier_flags.mean() * 100,
})
display(outlier_summary)

,count,mean,std,min,1%,25%,50%,75%,99%,max,skewness
floor_area_sqm,221629.000,96.777,24.028,31.000,47.000,81.000,93.000,112.000,151.000,366.700,0.257
resale_price,221629.000,524482.921,186642.033,140000.000,230000.000,385000.000,492888.000,630000.000,1070000.000,1658888.000,0.959
remaining_lease_years,221629.000,74.241,14.230,40.083,46.250,62.500,74.000,88.667,95.250,97.750,-0.020
storey_ordinal,221629.000,2.256,1.980,0.000,0.000,1.000,2.000,3.000,9.000,16.000,1.606
month_index,221629.000,56.215,30.163,0.000,1.000,31.000,57.000,82.000,106.000,107.000,-0.120
lease_commence_date,221629.000,1996.426,14.299,1966.000,1969.000,1985.000,1997.000,2012.000,2020.000,2021.000,0.065


,lower_fence,upper_fence,lower_tail_count,upper_tail_count,outlier_count,outlier_pct
floor_area_sqm,34.500,158.500,101,1085,1186,0.535
resale_price,17500.000,997500.000,0,4051,4051,1.828
remaining_lease_years,23.250,127.917,0,0,0,0.000
storey_ordinal,-2.000,6.000,0,8206,8206,3.703
month_index,-45.500,158.500,0,0,0,0.000
lease_commence_date,1944.500,2052.500,0,0,0,0.000


In [15]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for column, ax in zip(numeric_columns, axes.flat):
    is_storey = column == "storey_ordinal"
    sns.histplot(data=df, x=column, bins=40, discrete=is_storey, stat="density", kde=True,
                 kde_kws={"cut": 0, "bw_adjust": 4 if is_storey else 1}, ax=ax)
    for quantile, label, color in [(0.25, "Q1", "#345995"), (0.5, "Median", "#d1495b"), (0.75, "Q3", "#e3a018")]:
        value = df[column].quantile(quantile)
        ax.axvline(value, linestyle="--", color=color, label=f"{label}: {value:,.2f}")
    ax.set_title(f"Distribution: {column}")
    ax.legend(fontsize=9)
plt.tight_layout()
save_figure(fig, "03_numeric_distributions.png")
plt.show()

display(df.nlargest(8, "floor_area_sqm")[list(raw.columns)])
display(df.nlargest(8, "resale_price")[list(raw.columns)])

/tmp/ipykernel_61370/1000997818.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
182823,2024-07,KALLANG/WHAMPOA,3 ROOM,53,JLN MA'MOR,01 TO 03,366.700,Terrace,1972,47 years,1568000.000
19693,2017-12,KALLANG/WHAMPOA,3 ROOM,65,JLN MA'MOR,01 TO 03,249.000,Terrace,1972,53 years 07 months,1053888.000
92512,2021-03,BISHAN,EXECUTIVE,446,BRIGHT HILL DR,07 TO 09,243.000,Maisonette,1990,68 years 01 month,1092888.000
94959,2021-04,BISHAN,EXECUTIVE,454,SIN MING AVE,10 TO 12,243.000,Maisonette,1989,67 years 08 months,1001000.000
107851,2021-09,KALLANG/WHAMPOA,3 ROOM,46,JLN BAHAGIA,01 TO 03,241.000,Terrace,1972,49 years 10 months,1235000.000
35797,2018-09,KALLANG/WHAMPOA,3 ROOM,41,JLN BAHAGIA,01 TO 03,237.000,Terrace,1972,52 years 10 months,1185000.000
95808,2021-04,KALLANG/WHAMPOA,3 ROOM,61,JLN TENTERAM,01 TO 03,222.000,Terrace,1972,50 years 03 months,1100000.000
8868,2017-06,KALLANG/WHAMPOA,3 ROOM,38,JLN BAHAGIA,01 TO 03,215.000,Terrace,1972,54 years 01 month,830000.000


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
212073,2025-06,QUEENSTOWN,5 ROOM,92,DAWSON RD,22 TO 24,122.000,Premium Apartment Loft,2016,89 years 11 months,1658888.000
199476,2025-11,BISHAN,5 ROOM,275A,BISHAN ST 24,34 TO 36,120.000,DBSS,2011,84 years 10 months,1632000.000
199498,2025-11,BISHAN,EXECUTIVE,135,BISHAN ST 12,22 TO 24,163.000,Maisonette,1987,60 years 08 months,1600000.000
202785,2025-08,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,49 TO 51,105.000,Type S2,2011,84 years 05 months,1600000.000
218475,2025-01,TOA PAYOH,5 ROOM,138A,LOR 1A TOA PAYOH,19 TO 21,117.000,DBSS,2012,86 years 06 months,1600000.000
202786,2025-09,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,40 TO 42,107.000,Type S2,2011,84 years 04 months,1590000.000
174171,2024-06,BUKIT MERAH,5 ROOM,9B,BOON TIONG RD,34 TO 36,112.000,Improved,2016,90 years 08 months,1588000.000
174172,2024-06,BUKIT MERAH,5 ROOM,96A,HENDERSON RD,46 TO 48,113.000,Improved,2019,94 years,1588000.000


In [16]:
type_outliers = df.groupby("flat_type")["floor_area_sqm"].agg(
    q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75), count="size"
)
type_outliers["lower_fence"] = type_outliers["q1"] - 1.5 * (type_outliers["q3"] - type_outliers["q1"])
type_outliers["upper_fence"] = type_outliers["q3"] + 1.5 * (type_outliers["q3"] - type_outliers["q1"])
within_type_area_outlier = (
    df["floor_area_sqm"].lt(df["flat_type"].map(type_outliers["lower_fence"]))
    | df["floor_area_sqm"].gt(df["flat_type"].map(type_outliers["upper_fence"]))
)
type_outliers["within_type_outlier_count"] = within_type_area_outlier.groupby(df["flat_type"]).sum()
display(type_outliers)

,q1,q3,count,lower_fence,upper_fence,within_type_outlier_count
flat_type,,,,,,
1 ROOM,31.000,31.000,79,31.000,31.000,0
2 ROOM,45.000,47.000,4486,42.000,50.000,628
3 ROOM,65.000,70.000,52785,57.500,77.500,4029
4 ROOM,91.000,101.000,93853,76.000,116.000,303
5 ROOM,112.000,122.000,54416,97.000,137.000,752
EXECUTIVE,141.000,148.000,15926,130.500,158.500,2951
MULTI-GENERATION,159.000,166.750,84,147.375,178.375,22


Price is strongly right-skewed, and its logarithm is nearly symmetric. Log price therefore reduces domination by the most expensive transactions and gives percentage-style coefficients. Log area does not reduce absolute skew in this sample; it is used later because its coefficient is an elasticity and because housing prices are plausibly multiplicative in size.

In [17]:
skewness_comparison = pd.DataFrame({
    "original": {
        "resale_price": df["resale_price"].skew(),
        "floor_area_sqm": df["floor_area_sqm"].skew(),
    },
    "log_transformed": {
        "resale_price": np.log(df["resale_price"]).skew(),
        "floor_area_sqm": np.log(df["floor_area_sqm"]).skew(),
    },
})
skewness_comparison["absolute_skew_before_minus_after"] = (
    skewness_comparison["original"].abs() - skewness_comparison["log_transformed"].abs()
)
display(skewness_comparison)

,original,log_transformed,absolute_skew_before_minus_after
resale_price,0.959,0.038,0.922
floor_area_sqm,0.257,-0.444,-0.187


## 7. Representation imbalance and common support

This regression dataset has no class label to balance. The relevant problem is unequal representation: Towns and housing categories contribute very different numbers of transactions, and some Town × Flat Type combinations are sparse.

The primary explanatory model retains actual transaction frequencies because it describes the published transactions. Artificial over-sampling would change that target population. Imbalance is managed by reporting support, retaining confidence intervals, smoothing high-cardinality addresses, evaluating group errors, and fitting an equal-Town weighted sensitivity model later.

In [18]:
imbalance_fields = ["town", "flat_type", "flat_model", "storey_range"]
imbalance_training = df.loc[df["transaction_year"].between(2018, 2024)]
imbalance_test = df.loc[df["transaction_year"].eq(2025)]
summary_rows = []
shift_rows = []

for field in imbalance_fields:
    counts = df[field].value_counts()
    train_share = imbalance_training[field].value_counts(normalize=True)
    test_share = imbalance_test[field].value_counts(normalize=True)
    levels = counts.index.union(test_share.index)
    shift = 100 * (test_share.reindex(levels, fill_value=0)
                   - train_share.reindex(levels, fill_value=0))
    summary_rows.append({
        "field": field,
        "levels": len(counts),
        "smallest_group": counts.min(),
        "largest_group": counts.max(),
        "levels_below_200": counts.lt(200).sum(),
        "unseen_in_2025": (~levels.isin(train_share.index)).sum(),
        "distribution_shift_pct": 50 * shift.abs().sum() / 100,
    })
    for level in levels:
        shift_rows.append({"field": field, "level": level, "shift_pp": shift[level]})

imbalance_summary = pd.DataFrame(summary_rows).set_index("field")
low_support = pd.concat([
    df[field].value_counts().loc[lambda s: s.lt(200)].rename_axis("level").reset_index(name="rows").assign(field=field)
    for field in imbalance_fields
], ignore_index=True).sort_values(["field", "rows"])
display(imbalance_summary)
display(low_support[["field", "level", "rows"]])

town_type_counts = pd.crosstab(df["town"], df["flat_type"])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(np.log10(town_type_counts + 1), cmap="Blues", ax=ax,
            cbar_kws={"label": "log10(transaction count + 1)"})
ax.set(title="Common support across Town and Flat Type", xlabel="Flat Type", ylabel="Town")
plt.tight_layout()
save_figure(fig, "04_representation_support.png")
plt.show()

display(Markdown(f"""
The smallest Flat Type groups are **1 ROOM ({df['flat_type'].eq('1 ROOM').sum():,} rows)** and **MULTI-GENERATION ({df['flat_type'].eq('MULTI-GENERATION').sum():,} rows)**. Their separate coefficients and test errors are too unstable for strong conclusions. No 2025 category is unseen in 2018–2024, so the chronological test does not require inventing a coefficient for a new label.
"""))

,levels,smallest_group,largest_group,levels_below_200,unseen_in_2025,distribution_shift_pct
field,,,,,,
town,26,542,18108,0,0,5.570
flat_type,7,79,93853,2,0,3.191
flat_model,21,19,79163,7,0,5.133
storey_range,17,19,50849,3,0,0.891


,field,level,rows
8,flat_model,Premium Maisonette,19
7,flat_model,Improved-Maisonette,30
6,flat_model,3Gen,66
5,flat_model,Multi Generation,84
4,flat_model,Terrace,114
3,flat_model,Premium Apartment Loft,121
2,flat_model,Type S2,186
1,flat_type,1 ROOM,79
0,flat_type,MULTI-GENERATION,84
11,storey_range,49 TO 51,19


/tmp/ipykernel_61370/2796562832.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



The smallest Flat Type groups are **1 ROOM (79 rows)** and **MULTI-GENERATION (84 rows)**. Their separate coefficients and test errors are too unstable for strong conclusions. No 2025 category is unseen in 2018–2024, so the chronological test does not require inventing a coefficient for a new label.


## 8. Descriptive context before modelling

The snapshot covers every expected month from January 2017 through December 2025. The correlation table is a screening device: it reports marginal associations before controls and includes one-hot category indicators. The target-derived address encoding is shown on the common 2018–2025 subset but is not independent evidence of causation.

In [19]:
monthly = df.groupby("transaction_date").agg(
    transactions=("resale_price", "size"),
    median_price=("resale_price", "median"),
    median_area=("floor_area_sqm", "median"),
    median_remaining_lease=("remaining_lease_years", "median"),
)
expected_months = pd.date_range("2017-01-01", "2025-12-01", freq="MS")
display(pd.Series({
    "observed_months": len(monthly),
    "expected_months": len(expected_months),
    "missing_months": len(expected_months.difference(monthly.index)),
    "month_index_min": df["month_index"].min(),
    "month_index_max": df["month_index"].max(),
}, name="Time coverage").to_frame())
display(monthly.head())
display(monthly.tail())

,Time coverage
observed_months,108
expected_months,108
missing_months,0
month_index_min,0
month_index_max,107


,transactions,median_price,median_area,median_remaining_lease
transaction_date,,,,
2017-01-01,1176,403000.000,95.000,71.542
2017-02-01,1080,415000.000,96.000,74.500
2017-03-01,1889,415000.000,95.000,74.583
2017-04-01,1821,406600.000,95.000,74.000
2017-05-01,1961,410000.000,96.000,74.583


,transactions,median_price,median_area,median_remaining_lease
transaction_date,,,,
2025-08-01,2202,622000.000,93.000,71.667
2025-09-01,2172,645000.000,93.000,72.250
2025-10-01,1338,620000.000,93.000,70.625
2025-11-01,1657,618000.000,93.000,71.083
2025-12-01,2022,635000.000,93.000,71.083


In [20]:
continuous_features = [
    "floor_area_sqm", "remaining_lease_years", "storey_ordinal", "month_index"
]
association_sample = df.loc[address_encoding.index]
association_features = pd.concat([
    association_sample[continuous_features],
    one_hot.loc[association_sample.index],
    association_sample[["address_te"]],
], axis=1)
price_associations = pd.DataFrame({
    "pearson": association_features.corrwith(association_sample["resale_price"], method="pearson"),
    "spearman": association_features.corrwith(association_sample["resale_price"], method="spearman"),
})
price_associations["absolute_pearson"] = price_associations["pearson"].abs()
display(price_associations.nlargest(15, "absolute_pearson").drop(columns="absolute_pearson"))

,pearson,spearman
address_te,0.807,0.788
floor_area_sqm,0.577,0.613
flat_type_3 ROOM,-0.461,-0.521
month_index,0.397,0.418
storey_ordinal,0.341,0.247
flat_type_EXECUTIVE,0.309,0.299
remaining_lease_years,0.306,0.328
flat_type_5 ROOM,0.304,0.336
flat_model_New Generation,-0.282,-0.315
flat_model_Maisonette,0.231,0.206


## 9. Sequential hedonic regression

The outcome is `log(resale_price)`. Month indicators absorb market-wide changes across observed months; log area is an elasticity; storey intervals enter as categories; and remaining lease is measured in ten-year units centred at 70 years. Standard errors are clustered by Town + Street + Block.

`MULTI-GENERATION` Flat Type and `Multi Generation` Flat Model identify the same records. The redundant model indicator is removed so every fitted design is full rank. BEDOK, 4 ROOM, Model A, and 01 TO 03 are the reference categories.

In [21]:
import statsmodels.api as sm
import patsy
from IPython.display import Markdown


def regression_data(data):
    """Create readable regression fields without changing the supplied sample."""
    columns = list(raw.columns) + ["transaction_year", "month_index", "storey_ordinal", "remaining_lease_years"]
    result = data[columns].copy()
    for column in ["town", "flat_type", "flat_model", "storey_range", "month", "block", "street_name"]:
        result[column] = result[column].astype(str)
    result["log_price"] = np.log(result["resale_price"])
    result["log_area"] = np.log(result["floor_area_sqm"])
    result["lease_10"] = (result["remaining_lease_years"] - 70) / 10
    result["time_years"] = (result["month_index"] - 48) / 12
    result["calendar_month"] = result["month_index"].mod(12) + 1
    result["building_id"] = result[["town", "street_name", "block"]].agg(" | ".join, axis=1)
    return result


hedonic = regression_data(df)
assert not hedonic.duplicated(subset=list(raw.columns)).any()
assert hedonic.index.intersection(lease_excluded.index).empty
assert hedonic.index.intersection(duplicate_excluded.index).empty
TOWN = "C(town, Treatment(reference='BEDOK'))"
TIME = "C(month)"
TYPE = "C(flat_type, Treatment(reference='4 ROOM'))"
MODEL = "C(flat_model, Treatment(reference='Model A'))"
STOREY = "C(storey_range, Treatment(reference='01 TO 03'))"
REDUNDANT_MODEL = f"{MODEL}[T.Multi Generation]"


def fit_hedonic(formula, data, clustered=True):
    """Fit OLS, omitting the model indicator absorbed by its identical flat type."""
    outcome, design = patsy.dmatrices(formula, data, return_type="dataframe")
    design_info = design.design_info
    if REDUNDANT_MODEL in design.columns:
        design = design.drop(columns=REDUNDANT_MODEL)
    model = sm.OLS(outcome.iloc[:, 0], design)
    if clustered:
        fit = model.fit(method="qr", cov_type="cluster", use_t=True,
                        cov_kwds={"groups": data.loc[design.index, "building_id"]})
    else:
        fit = model.fit(method="qr")
    assert fit.model.rank == design.shape[1]
    return fit, design_info


def predict_log(fit, design_info, data):
    design = patsy.build_design_matrices([design_info], data, return_type="dataframe")[0]
    return pd.Series(fit.predict(design[fit.params.index]), index=data.index)


display(pd.Series({
    "explanatory_rows": len(hedonic),
    "towns": hedonic["town"].nunique(),
    "registration_months": hedonic["month"].nunique(),
    "building_clusters": hedonic["building_id"].nunique(),
}, name="Regression sample").to_frame())

,Regression sample
explanatory_rows,221629
towns,26
registration_months,108
building_clusters,9670


In [22]:
formulas = {
    "M0": f"log_price ~ {TOWN}",
    "M1": f"log_price ~ {TOWN} + {TIME}",
    "M2": f"log_price ~ {TOWN} + {TIME} + log_area",
    "M3": f"log_price ~ {TOWN} + {TIME} + log_area + {TYPE} + {MODEL}",
    "M4": f"log_price ~ {TOWN} + {TIME} + log_area + {TYPE} + {MODEL} + {STOREY} + lease_10",
}
main_fits = {}
main_info = {}
for name, formula in formulas.items():
    main_fits[name], main_info[name] = fit_hedonic(formula, hedonic)

model_comparison = pd.DataFrame({
    name: {
        "rows": fit.nobs,
        "coefficients": len(fit.params),
        "adjusted_R2": fit.rsquared_adj,
        "in_sample_log_RMSE": np.sqrt(np.mean(fit.resid ** 2)),
    }
    for name, fit in main_fits.items()
}).T
assert model_comparison["rows"].eq(len(hedonic)).all()
display(model_comparison)

,rows,coefficients,adjusted_R2,in_sample_log_RMSE
M0,221629.000,26.000,0.084,0.332
M1,221629.000,133.000,0.276,0.295
M2,221629.000,134.000,0.746,0.175
M3,221629.000,159.000,0.837,0.140
M4,221629.000,176.000,0.918,0.099


### 9.1 Redundancy and coefficient dependence

The exact redundant indicator is removed before fitting. Standardized VIFs check dependence among the continuous terms; movement across M0–M4 is then used to show the substantive dependence between Town and the housing composition variables.

In [23]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_fields = ["log_area", "lease_10", "storey_ordinal", "time_years"]
vif_design = hedonic[vif_fields].apply(lambda s: (s - s.mean()) / s.std())
vif_design = sm.add_constant(vif_design)
continuous_vif = pd.Series(
    [variance_inflation_factor(vif_design.to_numpy(), i) for i in range(1, vif_design.shape[1])],
    index=vif_fields,
    name="VIF",
).to_frame()
display(continuous_vif)
display(Markdown(f"""
The design-rank check confirms that every fitted main model is full rank after the exact redundant indicator is removed. The largest continuous-term VIF is **{continuous_vif['VIF'].max():.2f}**. This does not make individual coefficients causal; the large movement of the area coefficient and Town premiums across control steps shows that the housing attributes remain substantively interrelated.
"""))

,VIF
log_area,1.025
lease_10,1.108
storey_ordinal,1.083
time_years,1.004



The design-rank check confirms that every fitted main model is full rank after the exact redundant indicator is removed. The largest continuous-term VIF is **1.11**. This does not make individual coefficients causal; the large movement of the area coefficient and Town premiums across control steps shows that the housing attributes remain substantively interrelated.


## 10. Main finding: the PASIR RIS premium reverses

Town coefficients are converted to percentage differences relative to BEDOK. The critical comparison is M0–M4, fitted on exactly the same rows.

The sequence provides a direct answer: PASIR RIS first appears more expensive than BEDOK, but the sign reverses when floor area is added. The calculation below reports how much of the M1-to-M4 coefficient movement occurs at each step. Because sequential attribution depends on control order, the percentage is descriptive rather than a causal decomposition.

A common-segment sensitivity then restricts the data to 4 ROOM, Model A transactions, which are well represented in both Towns. This checks whether the adjusted result is merely extrapolation between different flat categories.

In [24]:
def town_effects(fit):
    confidence = fit.conf_int()
    rows = [{"town": "BEDOK", "premium_pct": 0., "low_pct": 0., "high_pct": 0.}]
    for term in fit.params.index:
        if term.startswith(TOWN + "[T."):
            rows.append({
                "town": term.split("[T.")[1][:-1],
                "premium_pct": 100 * np.expm1(fit.params[term]),
                "low_pct": 100 * np.expm1(confidence.loc[term, 0]),
                "high_pct": 100 * np.expm1(confidence.loc[term, 1]),
            })
    return pd.DataFrame(rows).set_index("town")


regional_effects = {name: town_effects(fit) for name, fit in main_fits.items()}
regional_path = pd.DataFrame({name: table["premium_pct"] for name, table in regional_effects.items()})
regional_change = pd.DataFrame({
    "M0_pct": regional_path["M0"],
    "M4_pct": regional_path["M4"],
})
regional_change["change_pp"] = regional_change["M4_pct"] - regional_change["M0_pct"]
display(regional_change.reindex(regional_change["change_pp"].abs().sort_values(ascending=False).index).head(10))

pasir_path = regional_path.loc["PASIR RIS", ["M0", "M1", "M2", "M3", "M4"]]
pasir_steps = pd.DataFrame({
    "model": pasir_path.index,
    "control_added": ["Town only", "Registration month", "Log floor area", "Flat Type and Model", "Storey and lease"],
    "premium_pct": pasir_path.values,
})
pasir_steps["change_from_previous_pp"] = pasir_steps["premium_pct"].diff()
composition_change = pasir_path["M4"] - pasir_path["M1"]
area_change = pasir_path["M2"] - pasir_path["M1"]
area_share_of_movement = abs(area_change / composition_change) * 100
display(pasir_steps)

focus_profiles = hedonic.loc[hedonic["town"].isin(["BEDOK", "PASIR RIS"])].groupby("town").agg(
    transactions=("resale_price", "size"),
    median_price=("resale_price", "median"),
    median_area=("floor_area_sqm", "median"),
    median_lease=("remaining_lease_years", "median"),
    five_room_share_pct=("flat_type", lambda s: 100 * s.eq("5 ROOM").mean()),
    executive_share_pct=("flat_type", lambda s: 100 * s.eq("EXECUTIVE").mean()),
)
display(focus_profiles)

common_segment = hedonic.loc[
    hedonic["flat_type"].eq("4 ROOM") & hedonic["flat_model"].eq("Model A")
]
common_formula = f"log_price ~ {TOWN} + {TIME} + log_area + {STOREY} + lease_10"
common_fit, _ = fit_hedonic(common_formula, common_segment)
common_term = TOWN + "[T.PASIR RIS]"
common_ci = common_fit.conf_int().loc[common_term]
common_effect = 100 * np.expm1(common_fit.params[common_term])
common_low, common_high = 100 * np.expm1(common_ci)
common_counts = common_segment.groupby("town").size().reindex(["BEDOK", "PASIR RIS"])
display(common_counts.rename("4 ROOM Model A rows").to_frame())

order = regional_effects["M4"]["premium_pct"].sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(16, 9), gridspec_kw={"width_ratios": [1.25, 1]})
for name, offset, color in [("M0", -.12, "#345995"), ("M4", .12, "#d1495b")]:
    effects = regional_effects[name].reindex(order)
    y = np.arange(len(order)) + offset
    axes[0].errorbar(
        effects["premium_pct"], y,
        xerr=[effects["premium_pct"] - effects["low_pct"], effects["high_pct"] - effects["premium_pct"]],
        fmt="o", markersize=4, capsize=2, color=color, label=name,
    )
axes[0].set_yticks(np.arange(len(order)), order)
axes[0].axvline(0, color="gray", linestyle="--")
axes[0].set(title="Town differences before and after controls", xlabel="Difference relative to BEDOK (%)")
axes[0].legend()

axes[1].plot(pasir_steps["model"], pasir_steps["premium_pct"], marker="o", linewidth=2)
axes[1].axhline(0, color="gray", linestyle="--")
for x, y in zip(pasir_steps["model"], pasir_steps["premium_pct"]):
    axes[1].annotate(f"{y:.1f}%", (x, y), xytext=(0, 8), textcoords="offset points", ha="center")
axes[1].set(title="PASIR RIS coefficient path", ylabel="Difference relative to BEDOK (%)")
plt.tight_layout()
save_figure(fig, "05_town_coefficient_path.png")
plt.show()

display(Markdown(f"""
**Conclusion.** PASIR RIS changes from **{pasir_path['M1']:.1f}%** after calendar controls to **{pasir_path['M4']:.1f}%** after all controls. Adding floor area at this step moves the estimate by **{area_change:.1f} percentage points** and produces the sign reversal. Under this control order, that step represents **{area_share_of_movement:.1f}%** of the total M1-to-M4 movement. PASIR RIS has a median floor area of **{focus_profiles.loc['PASIR RIS', 'median_area']:.0f} sqm**, versus **{focus_profiles.loc['BEDOK', 'median_area']:.0f} sqm** in BEDOK.

The common 4 ROOM, Model A segment contains **{common_counts['PASIR RIS']:,} PASIR RIS** and **{common_counts['BEDOK']:,} BEDOK** transactions. Within this more comparable segment, the adjusted PASIR RIS difference is **{common_effect:.1f}%** (95% CI **{common_low:.1f}% to {common_high:.1f}%**). The evidence supports a **housing-composition reversal**. Area produces the largest stepwise change in the original sequence, but area and Flat Type/Model share information; that sequence cannot identify their independent contributions. Section 10.3 examines this order dependence. This is an omitted-variable reversal; it is not presented as Simpson's paradox or as a causal location effect.
"""))

,M0_pct,M4_pct,change_pp
town,,,
PASIR RIS,25.954,-12.471,-38.424
PUNGGOL,17.411,-17.182,-34.592
SENGKANG,13.777,-19.294,-33.071
SEMBAWANG,7.334,-24.133,-31.466
CHOA CHU KANG,5.421,-24.665,-30.085
BISHAN,50.236,20.918,-29.317
BUKIT PANJANG,7.165,-18.890,-26.055
WOODLANDS,1.505,-23.174,-24.679
TAMPINES,19.794,-4.134,-23.928


,model,control_added,premium_pct,change_from_previous_pp
0,M0,Town only,25.954,NaN
1,M1,Registration month,26.508,0.554
2,M2,Log floor area,-6.251,-32.759
3,M3,Flat Type and Model,-10.234,-3.984
4,M4,Storey and lease,-12.471,-2.236


,transactions,median_price,median_area,median_lease,five_room_share_pct,executive_share_pct
town,,,,,,
BEDOK,11543,428000.000,88.000,59.583,18.124,4.981
PASIR RIS,6385,565000.000,122.000,71.750,31.511,25.607


,4 ROOM Model A rows
town,
BEDOK,1173
PASIR RIS,2373


/tmp/ipykernel_61370/297134839.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



**Conclusion.** PASIR RIS changes from **26.5%** after calendar controls to **-12.5%** after all controls. Adding floor area at this step moves the estimate by **-32.8 percentage points** and produces the sign reversal. Under this control order, that step represents **84.0%** of the total M1-to-M4 movement. PASIR RIS has a median floor area of **122 sqm**, versus **88 sqm** in BEDOK.

The common 4 ROOM, Model A segment contains **2,373 PASIR RIS** and **1,173 BEDOK** transactions. Within this more comparable segment, the adjusted PASIR RIS difference is **-16.1%** (95% CI **-18.0% to -14.2%**). The evidence supports a **housing-composition reversal**. Area produces the largest stepwise change in the original sequence, but area and Flat Type/Model share information; that sequence cannot identify their independent contributions. Section 10.3 examines this order dependence. This is an omitted-variable reversal; it is not presented as Simpson's paradox or as a causal location effect.


### 10.1 Geographic view of the same result

The two maps display M0 and M4 on the same colour scale. They visualize the coefficient change using planning-area proxies; they do not locate individual transactions or redefine HDB Town boundaries.

In [25]:
premium_map = town_boundaries.merge(regional_path[["M0", "M4"]], left_on="map_town", right_index=True, how="left")
limit = np.abs(regional_path[["M0", "M4"]].to_numpy()).max()
fig, axes = plt.subplots(1, 2, figsize=(17, 8))
for name, ax in zip(["M0", "M4"], axes):
    premium_map.plot(column=name, cmap="RdBu_r", vmin=-limit, vmax=limit,
                     legend=True, ax=ax, edgecolor="white", linewidth=.6,
                     missing_kwds={"color": "#e0e0e0"},
                     legend_kwds={"label": "Premium relative to BEDOK (%)", "shrink": .65})
    ax.set_title(f"{name}: town premiums on planning-area proxies")
    ax.set_axis_off()
plt.tight_layout()
save_figure(fig, "06_town_premium_maps.png")
plt.show()

/tmp/ipykernel_61370/3524415875.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 10.2 Common-support sensitivity: are the two towns comparable?

This local extension checks the scope of the PASIR RIS–BEDOK comparison before attributing the reversal to a particular housing attribute. The original all-town M4 remains the reference. We then estimate a two-town model, restrict to 4 ROOM / Model A, and investigate overlap in area, lease, storey and registration year.

Two complementary restrictions are used within that housing segment:

1. **Marginal intersection:** retain area and remaining lease within the intersection of the two towns' respective 5th–95th percentile ranges. This removes unsupported marginal tails but does not ensure joint overlap.
2. **Joint support:** form cells from registration year × storey interval × 5-sqm area bin × 5-year remaining-lease bin. Retain cells with at least one transaction in each town, then separately at least five. Bins are left-closed, anchored at zero; their widths and thresholds were fixed before examining the restricted regression results. Eligibility never uses price.

All restricted models retain month controls and the relevant original M4 housing controls, with public-building-clustered confidence intervals. Restricting the sample changes the target population and can change common slopes. The two-town Model A result therefore need not equal the preceding all-town Model A result.

Transaction weights are retained; this is **not exact matching or equal-cell weighting**. Report sample retention and residual attribute differences as well as the coefficient. Standardized mean differences use the same pooled SD from the unrestricted two-town 4 ROOM / Model A segment for every restriction. The analysis is exploratory, since the towns and housing segment follow an already inspected finding.


In [26]:
def section10_check_common_support():
    towns = ['BEDOK', 'PASIR RIS']
    colors = {'BEDOK': '#23658b', 'PASIR RIS': '#cc633d'}
    pair = hedonic.loc[hedonic['town'].isin(towns)].copy()
    pair['storey_lower'] = pair['storey_range'].str.extract(r'^(\d+)')[0].astype(int)
    segment = pair.loc[pair['flat_type'].eq('4 ROOM') & pair['flat_model'].eq('Model A')].copy()
    fields = ['floor_area_sqm', 'remaining_lease_years', 'storey_lower', 'transaction_year']
    quantiles = {.05: 'p05', .5: 'median', .95: 'p95'}
    profiles = []
    for label, sample in [('All types', pair), ('4 ROOM / Model A', segment)]:
        grouped = sample.groupby('town')
        profile = grouped[fields].quantile([.05, .5, .95]).unstack()
        profile.columns = [f'{field}_{quantiles[q]}' for field, q in profile.columns]
        profiles.append(profile.assign(sample=label, rows=grouped.size()).reset_index())
    profiles = pd.concat(profiles, ignore_index=True)
    profiles = profiles[['sample', 'town', 'rows'] + [f'{f}_{q}' for f in fields for q in ['p05', 'median', 'p95']]]
    display(profiles)

    fig, axes = plt.subplots(2, 4, figsize=(17.5, 7.8))
    for r, (label, sample) in enumerate([('All types', pair), ('4 ROOM / Model A', segment)]):
        for c, field in enumerate(fields):
            ax = axes[r, c]
            for town in towns:
                values = np.sort(sample.loc[sample['town'].eq(town), field].to_numpy())
                ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post',
                        color=colors[town], label=f'{town} (n={len(values):,})')
            ax.set(title=f'{label}: {field}', xlabel=field, ylabel='Cumulative proportion')
            ax.legend(fontsize=7)
    fig.suptitle('Same housing labels do not imply identical housing attributes')
    plt.tight_layout()
    save_figure(fig, "07_common_support_distributions.png")
    plt.show()

    mask = pd.Series(True, index=segment.index)
    bounds = []
    for field in ['floor_area_sqm', 'remaining_lease_years']:
        grouped = segment.groupby('town')[field]
        low, high = grouped.quantile(.05).max(), grouped.quantile(.95).min()
        bounds.append({'field': field, 'lower': low, 'upper': high})
        mask &= segment[field].between(low, high)
    marginal = segment.loc[mask].copy()
    bounds = pd.DataFrame(bounds)
    display(bounds)

    keys = pd.DataFrame({
        'year': segment['transaction_year'], 'storey': segment['storey_range'],
        'area_5sqm': np.floor(segment['floor_area_sqm'] / 5).astype(int),
        'lease_5years': np.floor(segment['remaining_lease_years'] / 5).astype(int),
    }, index=segment.index)
    cells = pd.crosstab([keys[c] for c in keys], segment['town']).reindex(columns=towns, fill_value=0)
    keys_index = pd.MultiIndex.from_frame(keys)
    joint = {n: segment.loc[keys_index.isin(cells.index[cells.min(axis=1).ge(n)])]
             for n in [1, 5]}
    sample_specs = [
        ('Full sample: original M4', hedonic, False),
        ('Two towns: all types', pair, False),
        ('Two towns: 4 ROOM Model A', segment, True),
        ('Segment: marginal range intersection', marginal, True),
        ('Segment: joint cells >=1 per town', joint[1], True),
        ('Segment: joint cells >=5 per town', joint[5], True),
    ]
    term = TOWN + '[T.PASIR RIS]'
    rows, balance_rows = [], []
    scale = {field: np.sqrt(sum(segment.loc[segment['town'].eq(t), field].var() for t in towns) / 2)
             for field in fields}
    for label, sample, same_type in sample_specs:
        if len(sample) < 100 or sample['building_id'].nunique() < 20:
            rows.append({'sample': label, 'rows': len(sample), 'status': 'Insufficient support; not fitted'})
            continue
        formula = f'log_price ~ {TOWN} + {TIME} + log_area + {STOREY} + lease_10'
        if not same_type:
            formula += f' + {TYPE} + {MODEL}'
        fit = main_fits['M4'] if label == 'Full sample: original M4' else fit_hedonic(formula, sample)[0]
        assert int(fit.nobs) == len(sample), 'No silent sample changes'
        low, high = fit.conf_int().loc[term]
        row = {'sample': label, 'rows': len(sample), 'buildings': sample['building_id'].nunique(),
               'BEDOK_rows': int(sample['town'].eq('BEDOK').sum()),
               'PASIR_RIS_rows': int(sample['town'].eq('PASIR RIS').sum()),
               'difference_pct': 100 * np.expm1(fit.params[term]),
               'low_pct': 100 * np.expm1(low), 'high_pct': 100 * np.expm1(high)}
        if same_type:
            for town in towns:
                row[town + '_retained_pct'] = 100 * sample['town'].eq(town).sum() / segment['town'].eq(town).sum()
            for field in fields:
                means = sample.groupby('town')[field].mean()
                balance_rows.append({'sample': label, 'field': field,
                    'BEDOK_mean': means['BEDOK'], 'PASIR_RIS_mean': means['PASIR RIS'],
                    'SMD_fixed_segment_SD': (means['PASIR RIS'] - means['BEDOK']) / scale[field]})
        rows.append(row)
    results = pd.DataFrame(rows)
    balance = pd.DataFrame(balance_rows)
    with pd.option_context('display.max_columns', None, 'display.width', 180):
        display(results)
        display(balance)
    plot = results.dropna(subset=['difference_pct']).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.errorbar(plot['difference_pct'], range(len(plot)),
                xerr=[plot['difference_pct'] - plot['low_pct'], plot['high_pct'] - plot['difference_pct']],
                fmt='o', color='#23658b', capsize=4)
    ax.set_yticks(range(len(plot)), [f"{r['sample']} (n={int(r['rows']):,})" for _, r in plot.iterrows()])
    ax.axvline(0, color='gray', linestyle='--')
    ax.set(title='Common-support sensitivity: populations differ across rows',
           xlabel='PASIR RIS relative to BEDOK (%); building-cluster 95% CI')
    plt.tight_layout()
    save_figure(fig, "08_common_support_sensitivity.png")
    plt.show()
    strict = results.loc[results['sample'].eq('Segment: joint cells >=5 per town')]
    if not strict.empty and strict['difference_pct'].notna().all():
        strict = strict.iloc[0]
        display(Markdown(f"""
**Support conclusion.** All fitted differences remain negative. The strict sample has
**{int(strict['rows']):,} transactions** (BEDOK **{int(strict['BEDOK_rows']):,}**; PASIR RIS **{int(strict['PASIR_RIS_rows']):,}**),
retaining **{strict['BEDOK_retained_pct']:.1f}%** and **{strict['PASIR RIS_retained_pct']:.1f}%** of the common-type samples.
Its adjusted difference is **{strict['difference_pct']:.2f}%** (95% CI **{strict['low_pct']:.2f}% to {strict['high_pct']:.2f}%**).
The conclusion applies to this narrower population; residual attribute differences remain.
"""))
    return {'profiles': profiles, 'bounds': bounds, 'cells': cells,
            'results': results, 'balance': balance}


section10_support = section10_check_common_support()


,sample,town,rows,floor_area_sqm_p05,floor_area_sqm_median,floor_area_sqm_p95,remaining_lease_years_p05,remaining_lease_years_median,remaining_lease_years_p95,storey_lower_p05,storey_lower_median,storey_lower_p95,transaction_year_p05,transaction_year_median,transaction_year_p95
0,All types,BEDOK,11543,60.000,88.000,141.000,52.250,59.583,92.167,1.000,7.000,13.000,2017.000,2021.000,2025.000
1,All types,PASIR RIS,6385,86.000,122.000,149.000,64.750,71.750,93.333,1.000,7.000,13.000,2017.000,2021.000,2025.000
2,4 ROOM / Model A,BEDOK,1173,87.000,94.000,108.000,60.933,83.750,94.867,1.000,7.000,16.000,2017.000,2021.000,2025.000
3,4 ROOM / Model A,PASIR RIS,2373,93.000,105.000,113.000,64.333,71.333,93.867,1.000,4.000,13.000,2017.000,2021.000,2025.000


/tmp/ipykernel_61370/3310952016.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,field,lower,upper
0,floor_area_sqm,93.000,108.000
1,remaining_lease_years,64.333,93.867


,sample,rows,buildings,BEDOK_rows,PASIR_RIS_rows,difference_pct,low_pct,high_pct,BEDOK_retained_pct,PASIR RIS_retained_pct
0,Full sample: original M4,221629,9670,11543,6385,-12.471,-13.346,-11.586,NaN,NaN
1,Two towns: all types,17928,928,11543,6385,-12.968,-14.160,-11.760,NaN,NaN
2,Two towns: 4 ROOM Model A,3546,340,1173,2373,-13.514,-15.334,-11.655,100.000,100.000
3,Segment: marginal range intersection,2378,297,521,1857,-16.060,-18.604,-13.437,44.416,78.255
4,Segment: joint cells >=1 per town,1660,282,494,1166,-14.436,-16.968,-11.827,42.114,49.136
5,Segment: joint cells >=5 per town,387,117,167,220,-10.810,-13.336,-8.211,14.237,9.271


,sample,field,BEDOK_mean,PASIR_RIS_mean,SMD_fixed_segment_SD
0,Two towns: 4 ROOM Model A,floor_area_sqm,97.262,104.051,1.124
1,Two towns: 4 ROOM Model A,remaining_lease_years,80.849,73.963,-0.655
2,Two towns: 4 ROOM Model A,storey_lower,7.302,5.756,-0.369
3,Two towns: 4 ROOM Model A,transaction_year,2021.489,2021.044,-0.182
4,Segment: marginal range intersection,floor_area_sqm,100.159,103.417,0.539
5,Segment: marginal range intersection,remaining_lease_years,79.812,73.766,-0.575
6,Segment: marginal range intersection,storey_lower,6.539,5.666,-0.208
7,Segment: marginal range intersection,transaction_year,2021.278,2020.908,-0.152
8,Segment: joint cells >=1 per town,floor_area_sqm,99.927,103.184,0.539
9,Segment: joint cells >=1 per town,remaining_lease_years,80.527,74.129,-0.609


/tmp/ipykernel_61370/3310952016.py:106: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



**Support conclusion.** All fitted differences remain negative. The strict sample has
**387 transactions** (BEDOK **167**; PASIR RIS **220**),
retaining **14.2%** and **9.3%** of the common-type samples.
Its adjusted difference is **-10.81%** (95% CI **-13.34% to -8.21%**).
The conclusion applies to this narrower population; residual attribute differences remain.


### 10.3 Control-order sensitivity: how much can be attributed to area?

The preceding approximately 84% statistic is the area step's share of the M1-to-M4 movement in percentage differences, under the original entry order. Correlated housing characteristics may absorb the same information, so this allocation can change when another block enters first.

Hold the full explanatory sample, BEDOK reference and registration-month controls fixed. Compare all six permutations of three housing blocks: **Area**, **Flat Type/Model**, and **Storey/Lease**. Each path uses exactly the same final M4 specification. Existing M1–M4 fits are reused; four additional subset specifications complete the eight possible housing-block subsets.

Report signed incremental changes and shares on both the exponentiated percentage-point scale and the original log-coefficient scale. Do not take absolute values or force nonnegative shares: a counter-movement is informative. Averaging allocations across orders is descriptive, not causal attribution. All paths must reconstruct the same total movement; their identical final coefficient follows mathematically from the common final model.

This supplement qualifies the earlier area-dominance interpretation without changing the original models, their saved results, or the prediction experiments.


In [27]:
def section10_check_control_order():
    from itertools import combinations, permutations

    blocks = {'Area': ['log_area'], 'Type/Model': [TYPE, MODEL],
              'Storey/Lease': [STOREY, 'lease_10']}
    names = list(blocks)
    term = TOWN + '[T.PASIR RIS]'
    existing = {frozenset(): main_fits['M1'],
                frozenset(['Area']): main_fits['M2'],
                frozenset(['Area', 'Type/Model']): main_fits['M3'],
                frozenset(names): main_fits['M4']}
    effects = {}
    for size in range(4):
        for subset in combinations(names, size):
            key = frozenset(subset)
            if key in existing:
                fit = existing[key]
            else:
                terms = [term for name in names if name in key for term in blocks[name]]
                formula = f'log_price ~ {TOWN} + {TIME} + ' + ' + '.join(terms)
                fit, _ = fit_hedonic(formula, hedonic)
            assert int(fit.nobs) == len(hedonic)
            assert np.array_equal(np.asarray(fit.model.data.row_labels), hedonic.index.to_numpy())
            beta = float(fit.params[term])
            effects[key] = {'beta_log': beta, 'difference_pct': 100 * np.expm1(beta)}
    start, end = effects[frozenset()], effects[frozenset(names)]
    total_pp = end['difference_pct'] - start['difference_pct']
    total_log = end['beta_log'] - start['beta_log']
    paths, contributions = [], []
    for order in permutations(names):
        label = ' -> '.join(order)
        paths.append({'order': label, 'step': 0, 'added': 'Time baseline', **start})
        selected, before = set(), start
        for step, block in enumerate(order, 1):
            selected.add(block)
            current = effects[frozenset(selected)]
            delta_pp = current['difference_pct'] - before['difference_pct']
            delta_log = current['beta_log'] - before['beta_log']
            contributions.append({'order': label, 'position': step, 'block': block,
                'change_pp': delta_pp, 'share_pp_pct': 100 * delta_pp / total_pp,
                'change_log': delta_log, 'share_log_pct': 100 * delta_log / total_log})
            paths.append({'order': label, 'step': step, 'added': block, **current})
            before = current
    paths = pd.DataFrame(paths)
    contributions = pd.DataFrame(contributions)
    assert np.allclose(contributions.groupby('order')['change_pp'].sum(), total_pp)
    assert np.allclose(contributions.groupby('order')['share_pp_pct'].sum(), 100)
    assert paths.loc[paths['step'].eq(3), 'difference_pct'].nunique() == 1
    summary = contributions.groupby('block').agg(
        min_share_pp_pct=('share_pp_pct', 'min'), max_share_pp_pct=('share_pp_pct', 'max'),
        mean_share_pp_pct=('share_pp_pct', 'mean'),
        min_share_log_pct=('share_log_pct', 'min'), max_share_log_pct=('share_log_pct', 'max')).reset_index()
    with pd.option_context('display.max_columns', None, 'display.width', 180):
        display(paths)
        display(contributions)
        display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    for label, part in paths.groupby('order', sort=False):
        axes[0].plot(part['step'], part['difference_pct'], marker='o', label=label)
    axes[0].axhline(0, color='gray', linestyle='--')
    axes[0].set(xticks=range(4), xlabel='Housing blocks added; time always controlled',
                ylabel='PASIR RIS relative to BEDOK (%)', title='Identical endpoints; different intermediate paths')
    axes[0].legend(fontsize=7)
    for j, block in enumerate(names):
        values = contributions.loc[contributions['block'].eq(block), 'share_pp_pct']
        axes[1].scatter(values, np.full(len(values), j), s=55, alpha=.65)
    axes[1].set_yticks(range(3), names)
    axes[1].axvline(0, color='gray', linestyle='--')
    axes[1].set(xlabel='Signed share of total percentage-point movement (%)',
                title='Attribution depends on order; not causal shares')
    plt.tight_layout()
    save_figure(fig, "09_control_order_sensitivity.png")
    plt.show()

    area = summary.set_index('block').loc['Area']
    original = contributions.loc[
        contributions['order'].eq('Area -> Type/Model -> Storey/Lease') & contributions['block'].eq('Area')
    ].iloc[0]
    type_first = effects[frozenset(['Type/Model'])]['difference_pct']
    display(Markdown(f"""
**Order conclusion.** Area accounts for **{original['share_pp_pct']:.2f}%** of the M1-to-M4 movement
in the original sequence, but **{area['min_share_pp_pct']:.2f}%–{area['max_share_pp_pct']:.2f}%** across orders
(log scale: **{area['min_share_log_pct']:.2f}%–{area['max_share_log_pct']:.2f}%**).
Type/Model first already reverses the difference from **{start['difference_pct']:.2f}% to {type_first:.2f}%**.
Housing composition matters, but area and housing type share information: 84% is order-specific,
not an independent or causal contribution. All paths end at **{end['difference_pct']:.2f}%**
because their final sample and model are identical, not because they independently confirm the result.
"""))
    return {'paths': paths, 'contributions': contributions, 'summary': summary}


section10_order = section10_check_control_order()


,order,step,added,beta_log,difference_pct
0,Area -> Type/Model -> Storey/Lease,0,Time baseline,0.235,26.508
1,Area -> Type/Model -> Storey/Lease,1,Area,-0.065,-6.251
2,Area -> Type/Model -> Storey/Lease,2,Type/Model,-0.108,-10.234
3,Area -> Type/Model -> Storey/Lease,3,Storey/Lease,-0.133,-12.471
4,Area -> Storey/Lease -> Type/Model,0,Time baseline,0.235,26.508
5,Area -> Storey/Lease -> Type/Model,1,Area,-0.065,-6.251
6,Area -> Storey/Lease -> Type/Model,2,Storey/Lease,-0.144,-13.403
7,Area -> Storey/Lease -> Type/Model,3,Type/Model,-0.133,-12.471
8,Type/Model -> Area -> Storey/Lease,0,Time baseline,0.235,26.508
9,Type/Model -> Area -> Storey/Lease,1,Type/Model,-0.103,-9.819


,order,position,block,change_pp,share_pp_pct,change_log,share_log_pct
0,Area -> Type/Model -> Storey/Lease,1,Area,-32.759,84.043,-0.300,81.362
1,Area -> Type/Model -> Storey/Lease,2,Type/Model,-3.984,10.221,-0.043,11.790
2,Area -> Type/Model -> Storey/Lease,3,Storey/Lease,-2.236,5.737,-0.025,6.849
3,Area -> Storey/Lease -> Type/Model,1,Area,-32.759,84.043,-0.300,81.362
4,Area -> Storey/Lease -> Type/Model,2,Storey/Lease,-7.152,18.349,-0.079,21.545
5,Area -> Storey/Lease -> Type/Model,3,Type/Model,0.932,-2.391,0.011,-2.907
6,Type/Model -> Area -> Storey/Lease,1,Type/Model,-36.327,93.197,-0.338,91.897
7,Type/Model -> Area -> Storey/Lease,2,Area,-0.416,1.067,-0.005,1.254
8,Type/Model -> Area -> Storey/Lease,3,Storey/Lease,-2.236,5.737,-0.025,6.849
9,Type/Model -> Storey/Lease -> Area,1,Type/Model,-36.327,93.197,-0.338,91.897


,block,min_share_pp_pct,max_share_pp_pct,mean_share_pp_pct,min_share_log_pct,max_share_log_pct
0,Area,1.067,84.043,43.074,1.254,81.362
1,Storey/Lease,-2.366,31.437,15.055,-2.762,27.659
2,Type/Model,-2.391,93.197,41.871,-2.907,91.897


/tmp/ipykernel_61370/4167526634.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



**Order conclusion.** Area accounts for **84.04%** of the M1-to-M4 movement
in the original sequence, but **1.07%–84.04%** across orders
(log scale: **1.25%–81.36%**).
Type/Model first already reverses the difference from **26.51% to -9.82%**.
Housing composition matters, but area and housing type share information: 84% is order-specific,
not an independent or causal contribution. All paths end at **-12.47%**
because their final sample and model are identical, not because they independently confirm the result.


## 11. What the main controls mean

The control variables are necessary to make Town comparisons more comparable. Their coefficients remain conditional associations. Flat Model is retained as a nuisance control but is not ranked individually because several labels have limited support and overlap strongly with Flat Type.

In [28]:
main = main_fits["M4"]
confidence = main.conf_int()
key_effects = pd.DataFrame({
    "effect": ["Area elasticity", "Ten-year remaining-lease difference"],
    "estimate": [main.params["log_area"], 100 * np.expm1(main.params["lease_10"])],
    "low": [confidence.loc["log_area", 0], 100 * np.expm1(confidence.loc["lease_10", 0])],
    "high": [confidence.loc["log_area", 1], 100 * np.expm1(confidence.loc["lease_10", 1])],
    "unit": ["% price per 1% area", "% price per 10 lease years"],
}).set_index("effect")
display(key_effects)

storey_rows = [{"storey_range": "01 TO 03", "difference_pct": 0., "low_pct": 0., "high_pct": 0.}]
for term in main.params.index:
    if term.startswith(STOREY + "[T."):
        label = term.split("[T.")[1][:-1]
        storey_rows.append({
            "storey_range": label,
            "difference_pct": 100 * np.expm1(main.params[term]),
            "low_pct": 100 * np.expm1(confidence.loc[term, 0]),
            "high_pct": 100 * np.expm1(confidence.loc[term, 1]),
        })
storey_effects = pd.DataFrame(storey_rows).set_index("storey_range").reindex(storey_order)
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(storey_effects))
ax.errorbar(
    storey_effects["difference_pct"], y,
    xerr=[storey_effects["difference_pct"] - storey_effects["low_pct"],
          storey_effects["high_pct"] - storey_effects["difference_pct"]],
    fmt="o", capsize=3,
)
ax.set_yticks(y, storey_effects.index)
ax.axvline(0, color="gray", linestyle="--")
ax.set(title="Adjusted Storey Range differences", xlabel="Difference relative to 01 TO 03 (%)")
plt.tight_layout()
save_figure(fig, "10_storey_effects.png")
plt.show()

,estimate,low,high,unit
effect,,,,
Area elasticity,0.811,0.778,0.844,% price per 1% area
Ten-year remaining-lease difference,10.392,10.155,10.629,% price per 10 lease years


/tmp/ipykernel_61370/759065614.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Diagnostics and decision sensitivity

Residual diagnostics assess functional-form limitations. The exclusion rules are mandatory for every downstream analytical and modelling frame, so they are audited rather than reversed in a sensitivity model. The remaining sensitivity compares categorical and ordinal representations of Storey Range. The main Town conclusion is considered robust if this reasonable encoding alternative does not materially change Town coefficients.


In [29]:
diagnostics = pd.DataFrame({
    "fitted_log_price": main.fittedvalues,
    "log_residual": main.resid,
}).sample(8000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=diagnostics, x="fitted_log_price", y="log_residual", alpha=.15, s=10, ax=axes[0])
axes[0].axhline(0, color="gray", linestyle="--")
axes[0].set_title("M4 residuals versus fitted values")
sm.qqplot(diagnostics["log_residual"], line="s", ax=axes[1])
axes[1].set_title("M4 sampled residual Q-Q plot")
plt.tight_layout()
save_figure(fig, "11_m4_diagnostics.png")
plt.show()

downstream_exclusion_audit = pd.Series({
    "duplicate_rows_in_hedonic_frame": int(hedonic.duplicated(subset=list(raw.columns)).sum()),
    "lease_excluded_indices_in_hedonic_frame": len(hedonic.index.intersection(lease_excluded.index)),
    "duplicate_excluded_indices_in_hedonic_frame": len(hedonic.index.intersection(duplicate_excluded.index)),
}, name="Downstream exclusion audit")
display(downstream_exclusion_audit.to_frame())
assert downstream_exclusion_audit.eq(0).all()

explanatory_variants = {
    "Categorical storey (main)": (formulas["M4"], hedonic),
    "Ordinal storey": (formulas["M4"].replace(STOREY, "storey_ordinal"), hedonic),
}
sensitivity_rows = []
for name, (formula, sample) in explanatory_variants.items():
    fit = main if name == "Categorical storey (main)" else fit_hedonic(formula, sample)[0]
    premiums = town_effects(fit)["premium_pct"]
    sensitivity_rows.append({
        "variant": name,
        "rows": fit.nobs,
        "PASIR_RIS_pct": premiums["PASIR RIS"],
        "maximum_Town_change_vs_main_pp": (premiums - regional_path["M4"]).abs().max(),
    })
sensitivity_table = pd.DataFrame(sensitivity_rows).set_index("variant")
display(sensitivity_table)
display(Markdown(f"""
All downstream exclusion audit counts are zero. Changing Storey Range from categorical intervals to an ordinal score changes the largest Town coefficient by **{sensitivity_table['maximum_Town_change_vs_main_pp'].max():.2f} percentage points**. PASIR RIS remains negative under both representations, so the sign reversal is not created by the Storey encoding choice.
"""))


/tmp/ipykernel_61370/2235155392.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Downstream exclusion audit
duplicate_rows_in_hedonic_frame,0
lease_excluded_indices_in_hedonic_frame,0
duplicate_excluded_indices_in_hedonic_frame,0


,rows,PASIR_RIS_pct,maximum_Town_change_vs_main_pp
variant,,,
Categorical storey (main),221629.000,-12.471,0.000
Ordinal storey,221629.000,-12.431,0.358



All downstream exclusion audit counts are zero. Changing Storey Range from categorical intervals to an ordinal score changes the largest Town coefficient by **0.36 percentage points**. PASIR RIS remains negative under both representations, so the sign reversal is not created by the Storey encoding choice.


## 13. Out-of-time preparation comparison

Use 2018–2023 for development, 2024 for time-trend selection, and 2018–2024 for final training. Every model is tested on the same cleaned 2025 records. All models use OLS; the variants change only data preparation. The 128 lease-inconsistent rows and 309 duplicate excess rows are excluded from every variant, so the comparison does not allow known integrity issues back into a baseline.

| Variant | Preparation represented |
|---|---|
| B0 minimal/source-like | Mandatory parsing and one-hot encoding; raw price/area, ordinal storey, linear lease/time |
| B1 + categorical structure | Add Flat Model and represent Storey Range as categories |
| B2 + functional forms | Log area and quadratic remaining lease; raw-price OLS |
| B3 + log target | B2 with log-price OLS and training-residual smearing |
| B4 + historical address | B3 plus leakage-controlled hierarchical address encoding |

B0 is minimally processed rather than literally unparsed because OLS cannot consume date and lease strings. The comparison reports MAE, RMSE, signed error, equal-Town macro-MAE, and worst-Town MAE.


In [30]:
forecast = hedonic.loc[hedonic["transaction_year"].ge(2018)].copy()
forecast["log_address_te"] = np.log(address_encoding.loc[forecast.index, "address_te"])

development = forecast.loc[forecast["transaction_year"].le(2023)]
validation = forecast.loc[forecast["transaction_year"].eq(2024)]
training = forecast.loc[forecast["transaction_year"].le(2024)]
test = forecast.loc[forecast["transaction_year"].eq(2025)]

for frame in [development, validation, training, test]:
    assert frame.index.intersection(lease_excluded.index).empty
    assert frame.index.intersection(duplicate_excluded.index).empty
for column in ["town", "flat_type", "flat_model", "storey_range"]:
    assert set(test[column]) <= set(training[column])
display(pd.DataFrame({
    "period": ["development", "validation", "final training", "test"],
    "years": ["2018–2023", "2024", "2018–2024", "2025"],
    "rows": [len(development), len(validation), len(training), len(test)],
}))


def price_scores(actual, predicted, towns):
    error = predicted - actual
    town_mae = error.abs().groupby(towns).mean()
    return {
        "MAE_SGD": error.abs().mean(),
        "RMSE_SGD": np.sqrt(np.mean(error ** 2)),
        "mean_error_SGD": error.mean(),
        "Town_macro_MAE_SGD": town_mae.mean(),
        "worst_Town_MAE_SGD": town_mae.max(),
    }


def predict_price(formula, fit_data, predict_data, log_target):
    fit, info = fit_hedonic(formula, fit_data, clustered=False)
    fitted = predict_log(fit, info, predict_data)
    if log_target:
        fitted = np.exp(fitted) * np.exp(fit.resid).mean()
    return fit, fitted


minimal_rhs = (
    f"{TOWN} + {TYPE} + floor_area_sqm + storey_ordinal "
    "+ remaining_lease_years + TIME_TERM"
)
structured_rhs = (
    f"{TOWN} + {TYPE} + {MODEL} + floor_area_sqm + {STOREY} "
    "+ remaining_lease_years + TIME_TERM"
)
prepared_rhs = (
    f"{TOWN} + {TYPE} + {MODEL} + log_area + {STOREY} "
    "+ lease_10 + I(lease_10 ** 2) + TIME_TERM"
)

trend_rows = []
for trend, term in {
    "linear": "time_years + C(calendar_month)",
    "quadratic": "time_years + I(time_years ** 2) + C(calendar_month)",
}.items():
    formula = "log_price ~ " + prepared_rhs.replace("TIME_TERM", term)
    _, predicted = predict_price(formula, development, validation, log_target=True)
    trend_rows.append({"trend": trend, **price_scores(
        validation["resale_price"], predicted, validation["town"]
    )})
trend_validation = pd.DataFrame(trend_rows).set_index("trend")
selected_trend = trend_validation["MAE_SGD"].idxmin()
time_term = {
    "linear": "time_years + C(calendar_month)",
    "quadratic": "time_years + I(time_years ** 2) + C(calendar_month)",
}[selected_trend]
display(trend_validation)
print(f"Selected using 2024 validation MAE: {selected_trend}")

pipeline_formulas = {
    "B0 minimal/source-like": "resale_price ~ " + minimal_rhs.replace("TIME_TERM", time_term),
    "B1 + categorical structure": "resale_price ~ " + structured_rhs.replace("TIME_TERM", time_term),
    "B2 + prepared features": "resale_price ~ " + prepared_rhs.replace("TIME_TERM", time_term),
    "B3 + log target": "log_price ~ " + prepared_rhs.replace("TIME_TERM", time_term),
    "B4 + historical address": "log_price ~ " + prepared_rhs.replace("TIME_TERM", time_term) + " + log_address_te",
}
development_sets = {name: development for name in pipeline_formulas}
training_sets = {name: training for name in pipeline_formulas}
log_variants = {"B3 + log target", "B4 + historical address"}
pipeline_predictions = pd.DataFrame(index=test.index)
pipeline_models = {}
result_rows = []

for name, formula in pipeline_formulas.items():
    _, validation_prediction = predict_price(
        formula, development_sets[name], validation, name in log_variants
    )
    fit, test_prediction = predict_price(
        formula, training_sets[name], test, name in log_variants
    )
    pipeline_models[name] = fit
    pipeline_predictions[name] = test_prediction
    result_rows.append({
        "variant": name,
        "training_rows": len(training_sets[name]),
        "test_rows": len(test),
        "validation_MAE_SGD": np.mean(abs(validation_prediction - validation["resale_price"])),
        **price_scores(test["resale_price"], test_prediction, test["town"]),
    })

pipeline_results = pd.DataFrame(result_rows).set_index("variant")
assert pipeline_results["test_rows"].eq(len(test)).all()
display(pipeline_results)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
pipeline_results[["MAE_SGD", "Town_macro_MAE_SGD"]].plot.bar(ax=axes[0])
axes[0].set(title="Preparation comparison on 2025", xlabel="", ylabel="Error (SGD)")
pipeline_results["mean_error_SGD"].plot.bar(ax=axes[1], color="#d1495b")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(title="Mean prediction error", xlabel="", ylabel="Predicted minus actual (SGD)")
plt.tight_layout()
save_figure(fig, "12_preparation_comparison.png")
plt.show()

minimal_mae = pipeline_results.loc["B0 minimal/source-like", "MAE_SGD"]
prepared_mae = pipeline_results.loc["B4 + historical address", "MAE_SGD"]
display(Markdown(f"""
**Prediction conclusion.** Systematic preparation reduces 2025 MAE from **SGD {minimal_mae:,.0f}** to **SGD {prepared_mae:,.0f}**, a reduction of **{100 * (1 - prepared_mae / minimal_mae):.1f}%**. The largest changes come from functional-form preparation, the log target, and address information. B4 is more accurate overall but its signed error and subgroup behavior must be considered below.
"""))


,period,years,rows
0,development,2018–2023,148486
1,validation,2024,27797
2,final training,2018–2024,176283
3,test,2025,25009


,MAE_SGD,RMSE_SGD,mean_error_SGD,Town_macro_MAE_SGD,worst_Town_MAE_SGD
trend,,,,,
linear,44125.319,59353.894,-9754.042,48616.771,74141.717
quadratic,67128.300,85009.789,56255.841,75911.926,134486.018


Selected using 2024 validation MAE: linear


,training_rows,test_rows,validation_MAE_SGD,MAE_SGD,RMSE_SGD,mean_error_SGD,Town_macro_MAE_SGD,worst_Town_MAE_SGD
variant,,,,,,,,
B0 minimal/source-like,176283,25009,57152.469,59788.767,85568.624,-17150.851,73376.603,182437.770
B1 + categorical structure,176283,25009,54302.426,57290.168,81432.205,-17128.048,68648.247,143272.271
B2 + prepared features,176283,25009,53972.249,56978.215,81021.820,-16715.140,68026.869,142645.433
B3 + log target,176283,25009,44125.319,47231.629,63323.880,329.726,51241.563,85006.510
B4 + historical address,176283,25009,42453.457,46297.934,62040.999,14603.157,48728.132,74160.857


/tmp/ipykernel_61370/2502200387.py:116: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



**Prediction conclusion.** Systematic preparation reduces 2025 MAE from **SGD 59,789** to **SGD 46,298**, a reduction of **22.6%**. The largest changes come from functional-form preparation, the log target, and address information. B4 is more accurate overall but its signed error and subgroup behavior must be considered below.


### 13.1 Uneven errors and imbalance mitigation

Overall MAE weights each transaction equally, so high-volume Towns influence it more. First compare B0 and B4 within Town and Flat Type. Then fit B4 with inverse-Town-frequency weights. This weighted model gives each Town equal total influence and directly tests whether transaction imbalance changes reliability. It changes the estimand, so both results are reported.

In [31]:
group_rows = []
for grouping in ["town", "flat_type"]:
    for group, part in test.groupby(grouping):
        for variant in ["B0 minimal/source-like", "B4 + historical address"]:
            error = pipeline_predictions.loc[part.index, variant] - part["resale_price"]
            group_rows.append({
                "grouping": grouping,
                "group": group,
                "rows": len(part),
                "variant": variant,
                "MAE_SGD": error.abs().mean(),
            })
group_errors = pd.DataFrame(group_rows).pivot(
    index=["grouping", "group", "rows"], columns="variant", values="MAE_SGD"
).reset_index()
group_errors["change_SGD"] = (
    group_errors["B4 + historical address"] - group_errors["B0 minimal/source-like"]
)
supported_groups = group_errors.loc[group_errors["rows"].ge(100)]
display(pd.concat([
    supported_groups.nsmallest(6, "change_SGD"),
    supported_groups.nlargest(6, "change_SGD"),
]).sort_values("change_SGD"))


def fit_weighted(formula, data, weights):
    outcome, design = patsy.dmatrices(formula, data, return_type="dataframe")
    info = design.design_info
    if REDUNDANT_MODEL in design.columns:
        design = design.drop(columns=REDUNDANT_MODEL)
    fit = sm.WLS(outcome.iloc[:, 0], design, weights=weights.loc[design.index]).fit()
    assert fit.model.rank == design.shape[1]
    return fit, info


town_frequency = training.groupby("town")["town"].transform("size")
town_weights = len(training) / (training["town"].nunique() * town_frequency)
balanced_fit, balanced_info = fit_weighted(
    pipeline_formulas["B4 + historical address"], training, town_weights
)
balanced_log = predict_log(balanced_fit, balanced_info, test)
balanced_smearing = np.average(
    np.exp(balanced_fit.resid), weights=town_weights.loc[balanced_fit.resid.index]
)
balanced_prediction = np.exp(balanced_log) * balanced_smearing

balance_results = pd.DataFrame({
    "Transaction-weighted B4": price_scores(
        test["resale_price"], pipeline_predictions["B4 + historical address"], test["town"]
    ),
    "Equal-Town weighted B4": price_scores(
        test["resale_price"], balanced_prediction, test["town"]
    ),
}).T
display(balance_results)

macro_reduction = (
    balance_results.loc["Transaction-weighted B4", "Town_macro_MAE_SGD"]
    - balance_results.loc["Equal-Town weighted B4", "Town_macro_MAE_SGD"]
)
bias_before = balance_results.loc["Transaction-weighted B4", "mean_error_SGD"]
bias_after = balance_results.loc["Equal-Town weighted B4", "mean_error_SGD"]
display(Markdown(f"""
**Imbalance conclusion.** Equal-Town weighting reduces Town macro-MAE by **SGD {macro_reduction:,.0f}** and changes mean overprediction from **SGD {bias_before:,.0f}** to **SGD {bias_after:,.0f}**. Unequal Town volume therefore affects predictive reliability. Transaction-weighted B4 answers performance for the average recorded sale; equal-Town B4 is preferable only when every Town should contribute equally to the objective.
"""))

variant,grouping,group,rows,B0 minimal/source-like,B4 + historical address,change_SGD
14,town,CENTRAL AREA,171,182437.770,50635.696,-131802.074
30,town,TOA PAYOH,1060,117738.283,58245.057,-59493.226
25,town,QUEENSTOWN,603,115720.662,62929.253,-52791.409
16,town,CLEMENTI,548,104489.782,62368.561,-42121.222
21,town,KALLANG/WHAMPOA,792,97419.937,58891.859,-38528.078
9,town,BISHAN,375,90626.059,53657.900,-36968.159
24,town,PUNGGOL,1535,44577.848,40160.065,-4417.783
8,town,BEDOK,1206,43775.056,42864.060,-910.996
15,town,CHOA CHU KANG,1026,36172.948,35382.998,-789.950
20,town,JURONG WEST,1588,41525.405,41408.506,-116.899


,MAE_SGD,RMSE_SGD,mean_error_SGD,Town_macro_MAE_SGD,worst_Town_MAE_SGD
Transaction-weighted B4,46297.934,62040.999,14603.157,48728.132,74160.857
Equal-Town weighted B4,46065.935,61480.916,7535.416,47264.778,69161.060



**Imbalance conclusion.** Equal-Town weighting reduces Town macro-MAE by **SGD 1,463** and changes mean overprediction from **SGD 14,603** to **SGD 7,535**. Unequal Town volume therefore affects predictive reliability. Transaction-weighted B4 answers performance for the average recorded sale; equal-Town B4 is preferable only when every Town should contribute equally to the objective.


### 13.2 Should valid price tails be removed?

Fit the same log-price specification with and without a training-only price-IQR restriction. Ordinal Storey is used in both versions because trimming removes the highest Storey category from training. The unchanged 2025 test set still includes expensive transactions.

In [32]:
price_q1, price_q3 = training["resale_price"].quantile([.25, .75])
price_upper = price_q3 + 1.5 * (price_q3 - price_q1)
tail_formula = pipeline_formulas["B3 + log target"].replace(STOREY, "storey_ordinal")
tail_training_sets = {
    "All training prices": training,
    "Training prices within IQR fence": training.loc[training["resale_price"].le(price_upper)],
}
tail_rows = []
for variant, sample in tail_training_sets.items():
    _, predicted = predict_price(tail_formula, sample, test, log_target=True)
    for test_group, mask in {
        "All test records": pd.Series(True, index=test.index),
        "Above training IQR fence": test["resale_price"].gt(price_upper),
    }.items():
        error = predicted.loc[mask] - test.loc[mask, "resale_price"]
        tail_rows.append({
            "training_policy": variant,
            "test_group": test_group,
            "test_rows": int(mask.sum()),
            "MAE_SGD": error.abs().mean(),
            "mean_error_SGD": error.mean(),
        })
tail_results = pd.DataFrame(tail_rows)
display(tail_results)

tail_wide = tail_results.pivot(index="test_group", columns="training_policy", values="MAE_SGD")
tail_increase = (
    tail_wide.loc["Above training IQR fence", "Training prices within IQR fence"]
    - tail_wide.loc["Above training IQR fence", "All training prices"]
)
display(Markdown(f"""
**Tail conclusion.** Removing high-price training records worsens MAE on expensive 2025 transactions by **SGD {tail_increase:,.0f}**. These prices represent a valid housing segment, so they remain in the prepared dataset even if trimming slightly changes an aggregate metric.
"""))

,training_policy,test_group,test_rows,MAE_SGD,mean_error_SGD
0,All training prices,All test records,25009,47379.511,295.405
1,All training prices,Above training IQR fence,1902,94737.760,-45696.726
2,Training prices within IQR fence,All test records,25009,47266.806,-1657.786
3,Training prices within IQR fence,Above training IQR fence,1902,98746.739,-50860.446



**Tail conclusion.** Removing high-price training records worsens MAE on expensive 2025 transactions by **SGD 4,009**. These prices represent a valid housing segment, so they remain in the prepared dataset even if trimming slightly changes an aggregate metric.


## 14. Conclusions

The notebook now supports one connected argument. The code below inserts the final values from this run.

In [33]:
pasir_m0 = regional_path.loc["PASIR RIS", "M0"]
pasir_m4 = regional_path.loc["PASIR RIS", "M4"]

display(Markdown(f"""
1. **Housing composition is important, but the area attribution is order-dependent.** The PASIR RIS difference relative to BEDOK changes from **{pasir_m0:.1f}%** in M0 to **{pasir_m4:.1f}%** in M4. Area produces the largest stepwise change in the original sequence, accounting for **{area_share_of_movement:.1f}%** of the M1-to-M4 movement. Across six entry orders, its share ranges from **{section10_order['summary'].set_index('block').loc['Area', 'min_share_pp_pct']:.2f}% to {section10_order['summary'].set_index('block').loc['Area', 'max_share_pp_pct']:.2f}%**; adding Flat Type/Model first already reverses the sign. Area and housing type share information, so this is not an independent or causal area contribution. The adjusted difference remains negative (**{common_effect:.1f}%**) in the common 4 ROOM, Model A segment. These are conditional associations, not a causal location effect or a claimed Simpson's paradox.

2. **Known integrity issues are excluded consistently.** The analytical frame removes **{len(lease_excluded):,} lease-inconsistent records** and **{len(duplicate_excluded):,} duplicate excess records**. Audit assertions confirm that none of their indices enters the hedonic, training, validation, or test frames.

3. **Systematic preparation improves out-of-time prediction, but not every reliability measure equally.** B0-to-B4 2025 MAE falls from **SGD {minimal_mae:,.0f}** to **SGD {prepared_mae:,.0f}**, or **{100 * (1 - prepared_mae / minimal_mae):.1f}%**. Some supported groups still worsen, and transaction-weighted B4 overpredicts by **SGD {bias_before:,.0f}** on average.

4. **Representation imbalance matters and can be managed without synthetic resampling.** Equal-Town weighting reduces Town macro-MAE by **SGD {macro_reduction:,.0f}** and changes average error from **SGD {bias_before:,.0f}** to **SGD {bias_after:,.0f}**. The weighted and unweighted results are both retained because they answer different population questions.

5. **Valid tails should not be deleted merely because an IQR rule flags them.** Training-tail deletion worsens error on expensive test transactions by **SGD {tail_increase:,.0f}**. This result supports retaining the price tail.
"""))



1. **Housing composition is important, but the area attribution is order-dependent.** The PASIR RIS difference relative to BEDOK changes from **26.0%** in M0 to **-12.5%** in M4. Area produces the largest stepwise change in the original sequence, accounting for **84.0%** of the M1-to-M4 movement. Across six entry orders, its share ranges from **1.07% to 84.04%**; adding Flat Type/Model first already reverses the sign. Area and housing type share information, so this is not an independent or causal area contribution. The adjusted difference remains negative (**-16.1%**) in the common 4 ROOM, Model A segment. These are conditional associations, not a causal location effect or a claimed Simpson's paradox.

2. **Known integrity issues are excluded consistently.** The analytical frame removes **128 lease-inconsistent records** and **309 duplicate excess records**. Audit assertions confirm that none of their indices enters the hedonic, training, validation, or test frames.

3. **Systematic preparation improves out-of-time prediction, but not every reliability measure equally.** B0-to-B4 2025 MAE falls from **SGD 59,789** to **SGD 46,298**, or **22.6%**. Some supported groups still worsen, and transaction-weighted B4 overpredicts by **SGD 14,603** on average.

4. **Representation imbalance matters and can be managed without synthetic resampling.** Equal-Town weighting reduces Town macro-MAE by **SGD 1,463** and changes average error from **SGD 14,603** to **SGD 7,535**. The weighted and unweighted results are both retained because they answer different population questions.

5. **Valid tails should not be deleted merely because an IQR rule flags them.** Training-tail deletion worsens error on expensive test transactions by **SGD 4,009**. This result supports retaining the price tail.


### Limitations

The adjusted Town coefficients remain associations. Renovation, view, exact accessibility, buyer characteristics, and detailed building condition are unavailable. Limited overlap between some Town and housing categories can still require extrapolation, although the common-segment check supports the PASIR RIS conclusion. Clustered standard errors address dependence within public building addresses but not all spatial dependence.

Prices are nominal SGD. Month indicators describe observed market time but do not identify inflation or a structural price index. The 2025 forecast can be affected by market drift, and historical target encoding can lag that drift. Town and Flat Type prediction comparisons describe reliability across housing groups, not demographic fairness because protected demographic attributes are absent.

The URA polygons are planning-area proxies. They support aggregate visualization only and do not geocode individual transactions.

## 15. Independent investigation of prediction errors

This extension investigates why BUKIT BATOK and TAMPINES deteriorate despite lower overall error, and why historical address encoding increases overprediction. All prediction variants and the 2024-selected time trend are inherited from Section 13. No hyperparameter is selected using 2025 outcomes. Subgroups were selected after reading saved test results, so this is an exploratory diagnosis rather than an independent confirmation.

Errors are predicted minus actual: positive signed error means overprediction. MAE measures magnitude and does not measure calibration. The comparison uses exactly the same cleaned 2025 test transactions. The addition was supported by OpenAI Codex; code execution, paired-record checks, reconstruction checks and table/figure inspection are used for verification. Group members must review and understand this extension before submission.

In [34]:
# Run after Section 14 of the upstream notebook. This code changes no upstream model.
from pathlib import Path
import json

ERROR_OUTPUT = Path('error_analysis_outputs')
ERROR_OUTPUT.mkdir(exist_ok=True)
FOCUS_TOWNS = ['BUKIT BATOK', 'TAMPINES']
VARIANTS = list(pipeline_predictions.columns)
SHORT = {v: v.split(' ')[0] for v in VARIANTS}
V0, V3, V4 = VARIANTS[0], VARIANTS[3], VARIANTS[4]
audit = test.copy()
for variant in VARIANTS:
    audit[f'pred_{SHORT[variant]}'] = pipeline_predictions[variant]
    audit[f'err_{SHORT[variant]}'] = pipeline_predictions[variant] - audit['resale_price']
assert audit.index.is_unique and len(audit) == len(test)
assert audit.filter(like='pred_').notna().all().all()

def save_table(name, table):
    table.to_csv(ERROR_OUTPUT / f'{name}.csv', index=False)
    return table

def metric_rows(data, label):
    rows = []
    for variant in VARIANTS:
        error = data[f'err_{SHORT[variant]}']
        rows.append(dict(group=label, variant=SHORT[variant], n=len(data),
            MAE_SGD=error.abs().mean(), RMSE_SGD=np.sqrt((error**2).mean()),
            signed_error_SGD=error.mean(), overprediction_pct=100*error.gt(0).mean(),
            mean_actual_SGD=data['resale_price'].mean()))
    return rows

stage_rows = metric_rows(audit, 'ALL')
for town in FOCUS_TOWNS:
    stage_rows += metric_rows(audit.loc[audit['town'].eq(town)], town)
stage_table = save_table('stage_metrics', pd.DataFrame(stage_rows))
display(stage_table)

# Paired bootstrap: aggregate differences by public building, then sample buildings.
# Keeps all records in each sampled building; does not account for every spatial/time dependency.
rng = np.random.default_rng(20261014)
paired_rows = []
for label, part in [('ALL', audit)] + [(t, audit.loc[audit['town'].eq(t)]) for t in FOCUS_TOWNS]:
    for base in ['B0', 'B3']:
        diff = part['err_B4'].abs() - part[f'err_{base}'].abs()
        clusters = pd.DataFrame({'building':part['building_id'], 'diff':diff}).groupby('building')['diff'].agg(['sum','count'])
        values = clusters.to_numpy()
        samples = []
        for _ in range(1000):
            sampled = values[rng.integers(0,len(values),len(values))]
            samples.append(sampled[:,0].sum()/sampled[:,1].sum())
        low, high = np.quantile(samples,[.025,.975])
        paired_rows.append(dict(group=label, comparison=f'B4 minus {base}', n=len(part),
            buildings=len(clusters), delta_MAE_SGD=diff.mean(), low_95_SGD=low, high_95_SGD=high))
paired_table = save_table('paired_building_bootstrap', pd.DataFrame(paired_rows))
display(paired_table)

monthly_rows = []
for town in FOCUS_TOWNS:
    for month, part in audit.loc[audit['town'].eq(town)].groupby('month'):
        monthly_rows += metric_rows(part, f'{town}|{month}')
monthly_table = pd.DataFrame(monthly_rows)
monthly_table[['town','month']] = monthly_table['group'].str.split('|',expand=True)
save_table('monthly_metrics', monthly_table)

# Within-Flat-Type contributions sum exactly to the town's MAE change.
type_rows = []
for town in FOCUS_TOWNS:
    town_test = audit.loc[audit['town'].eq(town)]
    history = training.loc[training['town'].eq(town)]
    recent = history.loc[history['transaction_year'].eq(2024)]
    for flat_type, part in town_test.groupby('flat_type'):
        h = history.loc[history['flat_type'].eq(flat_type)]
        r = recent.loc[recent['flat_type'].eq(flat_type)]
        delta = part['err_B4'].abs().mean()-part['err_B0'].abs().mean()
        type_rows.append(dict(town=town, flat_type=flat_type, n_2025=len(part),
            share_2025_pct=100*len(part)/len(town_test),
            share_training_pct=100*len(h)/len(history), share_2024_pct=100*len(r)/len(recent),
            MAE_B0=part['err_B0'].abs().mean(), MAE_B3=part['err_B3'].abs().mean(), MAE_B4=part['err_B4'].abs().mean(),
            mean_error_B0=part['err_B0'].mean(), mean_error_B3=part['err_B3'].mean(), mean_error_B4=part['err_B4'].mean(),
            contribution_to_town_delta_SGD=delta*len(part)/len(town_test),
            median_area_2024=r['floor_area_sqm'].median(), median_area_2025=part['floor_area_sqm'].median(),
            median_lease_2024=r['remaining_lease_years'].median(), median_lease_2025=part['remaining_lease_years'].median(),
            median_price_2024=r['resale_price'].median(), median_price_2025=part['resale_price'].median()))
type_table = save_table('flat_type_diagnosis',pd.DataFrame(type_rows))
for town in FOCUS_TOWNS:
    p = audit.loc[audit['town'].eq(town)]
    assert np.isclose(type_table.loc[type_table['town'].eq(town),'contribution_to_town_delta_SGD'].sum(),
                      p['err_B4'].abs().mean()-p['err_B0'].abs().mean())
display(type_table)

# Address history coverage is a support diagnostic, not a claim of causation.
reference = df.loc[df['transaction_year'].lt(2025)]
counts = reference.groupby(address_columns).size().rename('historical_n').reset_index()
support = audit[address_columns].merge(counts,on=address_columns,how='left',sort=False)
audit['historical_n'] = support['historical_n'].fillna(0).to_numpy()
audit['support_band'] = pd.cut(audit['historical_n'],[-1,0,9,29,99,np.inf],
                              labels=['unseen','1-9','10-29','30-99','100+'])
support_rows = []
for label, part in [('ALL',audit)] + [(t,audit.loc[audit['town'].eq(t)]) for t in FOCUS_TOWNS]:
    for band, sub in part.groupby('support_band',observed=True):
        support_rows.append(dict(group=label,support=str(band),n=len(sub),
            MAE_B3=sub['err_B3'].abs().mean(),MAE_B4=sub['err_B4'].abs().mean(),
            signed_error_B4=sub['err_B4'].mean()))
support_table = save_table('historical_address_support',pd.DataFrame(support_rows))
display(support_table)

# Exact decomposition of B4 minus B3 log predictions, centered at training-average encoding.
fit3, fit4 = pipeline_models[V3], pipeline_models[V4]
_, design3 = patsy.dmatrices(pipeline_formulas[V3],training,return_type='dataframe')
_, design4 = patsy.dmatrices(pipeline_formulas[V4],training,return_type='dataframe')
x3 = patsy.build_design_matrices([design3.design_info],test,return_type='dataframe')[0][fit3.params.index]
x4 = patsy.build_design_matrices([design4.design_info],test,return_type='dataframe')[0][fit4.params.index]
common = fit3.params.index
change = x3.mul(fit4.params.reindex(common)-fit3.params,axis=1)
center = training['log_address_te'].mean()
smear3, smear4 = np.exp(fit3.resid).mean(), np.exp(fit4.resid).mean()
components = pd.DataFrame(index=test.index)
components['address_centered'] = fit4.params['log_address_te']*(test['log_address_te']-center)
components['intercept_and_smearing'] = change['Intercept']+fit4.params['log_address_te']*center+np.log(smear4/smear3)
town_terms = [c for c in common if c.startswith(TOWN)]
time_terms = [c for c in common if c.startswith('C(calendar_month)') or 'time_years' in c]
other = [c for c in common if c not in town_terms+time_terms+['Intercept']]
components['town_coefficient_change'] = change[town_terms].sum(axis=1)
components['time_coefficient_change'] = change[time_terms].sum(axis=1)
components['flat_characteristic_change'] = change[other].sum(axis=1)
expected = np.log(audit['pred_B4']/audit['pred_B3'])
max_reconstruction_error = float((components.sum(axis=1)-expected).abs().max())
assert max_reconstruction_error < 1e-9
decomposition_rows = []
for label, idx in [('ALL',audit.index)] + [(t,audit.index[audit['town'].eq(t)]) for t in FOCUS_TOWNS]:
    row = dict(group=label,n=len(idx), **components.loc[idx].mean().mul(100).to_dict())
    row['total_log_prediction_change_x100'] = 100*expected.loc[idx].mean()
    row['mean_prediction_change_SGD'] = (audit.loc[idx,'pred_B4']-audit.loc[idx,'pred_B3']).mean()
    decomposition_rows.append(row)
decomposition_table = save_table('prediction_change_decomposition',pd.DataFrame(decomposition_rows))
display(decomposition_table)

# Fixed-model sensitivity: remove the latest history year, without fitting to 2025 outcomes.
# This diagnoses sensitivity to historical information; it is not a deployable tuned policy.
history_2023 = df.loc[df['transaction_year'].le(2023)]
frozen_encoding = hierarchical_target_encode(history_2023,test[address_columns])
frozen_test = test.copy()
frozen_test['log_address_te'] = np.log(frozen_encoding['address_te'])
frozen_x4 = patsy.build_design_matrices([design4.design_info],frozen_test,return_type='dataframe')[0][fit4.params.index]
frozen_pred = np.exp(fit4.predict(frozen_x4))*smear4
sensitivity_rows = []
for label, part in [('ALL',audit)] + [(t,audit.loc[audit['town'].eq(t)]) for t in FOCUS_TOWNS]:
    for policy,pred in [('B3 no address',part['pred_B3']),('B4 history through 2024',part['pred_B4']),
                        ('B4 fixed model / history through 2023',frozen_pred.loc[part.index])]:
        error = pred-part['resale_price']
        sensitivity_rows.append(dict(group=label,policy=policy,n=len(part),MAE_SGD=error.abs().mean(),
            signed_error_SGD=error.mean(),mean_prediction_SGD=pred.mean()))
sensitivity_table = save_table('frozen_history_sensitivity',pd.DataFrame(sensitivity_rows))
display(sensitivity_table)

encoding_shift = []
for year,part in forecast.groupby('transaction_year'):
    encoding_shift.append(dict(year=int(year),n=len(part),mean_log_address=part['log_address_te'].mean(),
        mean_price=part['resale_price'].mean()))
save_table('encoding_by_year',pd.DataFrame(encoding_shift))

fig,axes=plt.subplots(1,3,figsize=(15,4.5))
for group,ax in zip(['ALL']+FOCUS_TOWNS,axes):
    part=stage_table.loc[stage_table['group'].eq(group)]
    ax.plot(part['variant'],part['MAE_SGD']/1000,marker='o',label='MAE')
    ax.plot(part['variant'],part['signed_error_SGD']/1000,marker='s',label='Signed error')
    ax.axhline(0,color='gray',lw=.8)
    ax.set(title=f'{group} (n={int(part.n.iloc[0]):,})',ylabel='SGD thousands',xlabel='Preparation variant')
    ax.legend(fontsize=8)
fig.suptitle('Overall improvement can conceal local deterioration')
fig.tight_layout()
fig.savefig(ERROR_OUTPUT/'01_stage_error.png',dpi=180,bbox_inches='tight')
plt.show()

fig,axes=plt.subplots(2,2,figsize=(13,8))
for i,town in enumerate(FOCUS_TOWNS):
    p=monthly_table.loc[monthly_table['town'].eq(town)]
    for variant in ['B0','B3','B4']:
        s=p.loc[p['variant'].eq(variant)].sort_values('month')
        months=s['month'].str[-2:].astype(int)
        axes[i,0].plot(months,s['MAE_SGD']/1000,marker='o',label=variant)
        axes[i,1].plot(months,s['signed_error_SGD']/1000,marker='o',label=variant)
    for ax in axes[i]:
        ax.set(xticks=range(1,13),xlabel='2025 registration month',ylabel='SGD thousands')
        ax.legend(fontsize=8)
    axes[i,0].set_title(f'{town}: MAE')
    axes[i,1].set_title(f'{town}: signed error')
    axes[i,1].axhline(0,color='gray',lw=.8)
fig.tight_layout()
fig.savefig(ERROR_OUTPUT/'02_monthly_error.png',dpi=180,bbox_inches='tight')
plt.show()

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for town,ax in zip(FOCUS_TOWNS,axes):
    p=type_table.loc[type_table['town'].eq(town)].sort_values('contribution_to_town_delta_SGD')
    colors=np.where(p['contribution_to_town_delta_SGD']>0,'#d1495b','#345995')
    ax.barh(p['flat_type'],p['contribution_to_town_delta_SGD']/1000,color=colors)
    ax.axvline(0,color='gray',lw=.8)
    ax.set(title=town,xlabel='Contribution to B4 minus B0 MAE (SGD thousands)')
fig.suptitle('Which Flat Types account for the town-level deterioration?')
fig.tight_layout()
fig.savefig(ERROR_OUTPUT/'03_flat_type_contribution.png',dpi=180,bbox_inches='tight')
plt.show()

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
cols=list(components.columns)
d=decomposition_table.set_index('group')[cols]*1.0
d.T.plot.bar(ax=axes[0],width=.8)
axes[0].axhline(0,color='gray',lw=.8)
axes[0].set(title='Exact B4 minus B3 prediction decomposition',ylabel='Mean log prediction change x 100',xlabel='')
axes[0].tick_params(axis='x',labelrotation=30,labelsize=7)
short_policy={'B3 no address':'B3','B4 history through 2024':'B4','B4 fixed model / history through 2023':'B4 frozen'}
s=sensitivity_table.assign(policy=sensitivity_table['policy'].map(short_policy))
s.pivot(index='group',columns='policy',values='signed_error_SGD').reindex(['ALL']+FOCUS_TOWNS).plot.bar(ax=axes[1])
axes[1].axhline(0,color='gray',lw=.8)
axes[1].set(title='Sensitivity to adding 2024 address history',ylabel='Mean prediction error (SGD)',xlabel='')
axes[1].tick_params(axis='x',labelrotation=0)
fig.tight_layout()
fig.savefig(ERROR_OUTPUT/'04_address_mechanism.png',dpi=180,bbox_inches='tight')
plt.show()

summary = dict(source_rows=len(raw),analysis_rows=len(df),test_rows=len(test),
    source_sha256=metadata['snapshot_sha256'],selected_time_trend=selected_trend,
    address_coefficient=float(fit4.params['log_address_te']),smearing_B3=float(smear3),smearing_B4=float(smear4),
    reconstruction_max_abs_error=max_reconstruction_error,
    monthly_deterioration={t:int((monthly_table.loc[monthly_table.town.eq(t)].pivot(index='month',columns='variant',values='MAE_SGD')['B4'] > monthly_table.loc[monthly_table.town.eq(t)].pivot(index='month',columns='variant',values='MAE_SGD')['B0']).sum()) for t in FOCUS_TOWNS})
(ERROR_OUTPUT/'validation_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
display(Markdown('''### Interpretation boundaries

The stage table identifies which preparation transition coincides with deterioration. Flat-Type contributions are an exact descriptive allocation of the observed MAE difference; they do not prove that a change in housing mix caused it. Monthly tables distinguish persistent errors from isolated months. Building-cluster bootstrap intervals describe sampling uncertainty within this selected test sample; they are exploratory and do not cover all spatial or market uncertainty.

The decomposition exactly reconstructs the difference in B4 and B3 predictions, including changes to all shared regression coefficients and the smearing correction. It is an algebraic explanation of model behavior, not a causal explanation of housing prices. Centering reallocates the constant between the address and intercept components without changing their sum.

Frozen-history sensitivity holds the trained B4 coefficients fixed and changes only its historical encoding. It tests how strongly recent historical prices move predictions. It is out of the original training-feature distribution and must not be selected as a better production policy using 2025 results. No test-set bias correction or new model selection is performed. A future confirmatory experiment should pre-specify a time-normalized historical encoding and evaluate it on a new chronological holdout.
'''))


,group,variant,n,MAE_SGD,RMSE_SGD,signed_error_SGD,overprediction_pct,mean_actual_SGD
0,ALL,B0,25009,59788.767,85568.624,-17150.851,43.996,652505.664
1,ALL,B1,25009,57290.168,81432.205,-17128.048,43.684,652505.664
2,ALL,B2,25009,56978.215,81021.820,-16715.140,44.364,652505.664
3,ALL,B3,25009,47231.629,63323.880,329.726,50.278,652505.664
4,ALL,B4,25009,46297.934,62040.999,14603.157,58.823,652505.664
5,BUKIT BATOK,B0,1298,38931.821,48929.082,11818.577,57.550,609210.005
6,BUKIT BATOK,B1,1298,34996.994,44733.756,9697.678,58.475,609210.005
7,BUKIT BATOK,B2,1298,35561.621,45660.813,9330.990,59.322,609210.005
8,BUKIT BATOK,B3,1298,38260.700,47182.417,19660.834,63.945,609210.005
9,BUKIT BATOK,B4,1298,59242.035,75697.116,46570.728,72.573,609210.005


,group,comparison,n,buildings,delta_MAE_SGD,low_95_SGD,high_95_SGD
0,ALL,B4 minus B0,25009,8017,-13490.833,-15215.360,-11686.011
1,ALL,B4 minus B3,25009,8017,-933.695,-1972.612,54.503
2,BUKIT BATOK,B4 minus B0,1298,341,20310.214,13470.042,27006.048
3,BUKIT BATOK,B4 minus B3,1298,341,20981.335,15167.835,26864.721
4,TAMPINES,B4 minus B0,1918,615,10872.986,3785.596,18228.917
5,TAMPINES,B4 minus B3,1918,615,13262.230,7842.952,18897.874


,town,flat_type,n_2025,share_2025_pct,share_training_pct,share_2024_pct,MAE_B0,...,contribution_to_town_delta_SGD,median_area_2024,median_area_2025,median_lease_2024,median_lease_2025,median_price_2024,median_price_2025
0,BUKIT BATOK,2 ROOM,79,6.086,2.510,6.597,81871.433,...,-2360.747,47.000,47.000,94.583,93.583,350000.000,375000.000
1,BUKIT BATOK,3 ROOM,335,25.809,29.257,24.653,31788.948,...,-1500.997,68.000,68.000,61.833,60.417,398000.000,420000.000
2,BUKIT BATOK,4 ROOM,546,42.065,42.937,46.817,36541.216,...,11620.882,93.000,93.000,93.500,91.708,600000.000,625000.000
3,BUKIT BATOK,5 ROOM,266,20.493,17.125,17.593,37358.137,...,10436.638,113.000,113.000,93.500,92.417,759000.000,812500.000
4,BUKIT BATOK,EXECUTIVE,72,5.547,8.172,4.340,48994.361,...,2114.438,147.000,146.000,61.333,60.083,860000.000,869444.000
5,TAMPINES,2 ROOM,13,0.678,0.721,0.766,55972.461,...,-284.061,46.000,47.000,88.583,88.750,357000.000,420000.000
6,TAMPINES,3 ROOM,351,18.300,21.253,21.450,30422.288,...,505.925,73.000,73.000,60.333,59.583,457500.000,490000.000
7,TAMPINES,4 ROOM,891,46.455,40.762,44.178,36439.973,...,7723.496,100.000,97.000,68.167,69.833,625000.000,668000.000
8,TAMPINES,5 ROOM,508,26.486,26.763,25.843,53730.644,...,4928.120,122.000,121.000,68.667,70.542,730000.000,811000.000
9,TAMPINES,EXECUTIVE,154,8.029,10.299,7.661,98503.162,...,-1999.628,146.000,146.000,63.542,62.333,905000.000,965000.000


,group,support,n,MAE_B3,MAE_B4,signed_error_B4
0,ALL,unseen,816,66905.740,66405.495,-8936.127
1,ALL,1-9,1927,53370.703,57544.537,19502.450
2,ALL,10-29,14443,44842.867,42948.269,14425.240
3,ALL,30-99,7674,47892.009,47253.951,15768.407
4,ALL,100+,149,57628.025,66182.142,37386.086
5,BUKIT BATOK,unseen,87,70239.475,62695.903,62078.586
6,BUKIT BATOK,1-9,57,31238.520,32169.502,15202.638
7,BUKIT BATOK,10-29,834,34837.364,52544.426,37964.655
8,BUKIT BATOK,30-99,320,39739.366,80580.950,70371.549
9,TAMPINES,unseen,53,37393.847,91019.383,90963.050


,group,n,address_centered,intercept_and_smearing,town_coefficient_change,time_coefficient_change,flat_characteristic_change,total_log_prediction_change_x100,mean_prediction_change_SGD
0,ALL,25009,4.294,52.936,2.336,-3.871,-53.950,1.745,14273.432
1,BUKIT BATOK,1298,3.799,52.936,5.430,-3.873,-54.833,3.458,26909.895
2,TAMPINES,1918,9.098,52.936,-0.144,-3.872,-55.392,2.625,21945.631


,group,policy,n,MAE_SGD,signed_error_SGD,mean_prediction_SGD
0,ALL,B3 no address,25009,47231.629,329.726,652835.390
1,ALL,B4 history through 2024,25009,46297.934,14603.157,667108.822
2,ALL,B4 fixed model / history through 2023,25009,45014.877,1786.994,654292.659
3,BUKIT BATOK,B3 no address,1298,38260.700,19660.834,628870.838
4,BUKIT BATOK,B4 history through 2024,1298,59242.035,46570.728,655780.733
5,BUKIT BATOK,B4 fixed model / history through 2023,1298,50480.516,33788.396,642998.401
6,TAMPINES,B3 no address,1918,42692.930,10540.931,722832.642
7,TAMPINES,B4 history through 2024,1918,55955.160,32486.562,744778.273
8,TAMPINES,B4 fixed model / history through 2023,1918,36666.955,1916.354,714208.065


/tmp/ipykernel_61370/3877323356.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_61370/3877323356.py:191: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_61370/3877323356.py:203: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_61370/3877323356.py:220: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation boundaries

The stage table identifies which preparation transition coincides with deterioration. Flat-Type contributions are an exact descriptive allocation of the observed MAE difference; they do not prove that a change in housing mix caused it. Monthly tables distinguish persistent errors from isolated months. Building-cluster bootstrap intervals describe sampling uncertainty within this selected test sample; they are exploratory and do not cover all spatial or market uncertainty.

The decomposition exactly reconstructs the difference in B4 and B3 predictions, including changes to all shared regression coefficients and the smearing correction. It is an algebraic explanation of model behavior, not a causal explanation of housing prices. Centering reallocates the constant between the address and intercept components without changing their sum.

Frozen-history sensitivity holds the trained B4 coefficients fixed and changes only its historical encoding. It tests how strongly recent historical prices move predictions. It is out of the original training-feature distribution and must not be selected as a better production policy using 2025 results. No test-set bias correction or new model selection is performed. A future confirmatory experiment should pre-specify a time-normalized historical encoding and evaluate it on a new chronological holdout.


### 15.1 Composition standardization and support concentration

Holding 2025 within-Flat-Type error differences fixed, replace 2025 Flat-Type shares with 2024 shares. This asks whether the observed deterioration is mainly an arithmetic consequence of different Flat-Type proportions. It does not control every housing attribute or identify a temporal causal effect. Separately allocate B4-minus-B3 MAE changes to historical-address-support bands; positive contributions mean worsening.

In [35]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
error_tables = Path('error_analysis_outputs')
error_types = pd.read_csv(error_tables / 'flat_type_diagnosis.csv')
error_support = pd.read_csv(error_tables / 'historical_address_support.csv')
mix_rows = []
for town, part in error_types.groupby('town'):
    assert np.isclose(part['share_2024_pct'].sum(), 100)
    delta = part['MAE_B4'] - part['MAE_B0']
    observed = (delta * part['share_2025_pct'] / 100).sum()
    standardized = (delta * part['share_2024_pct'] / 100).sum()
    mix_rows.append({'town': town, 'observed_delta_MAE_SGD': observed,
        'delta_MAE_using_2024_flat_type_shares_SGD': standardized,
        'composition_difference_SGD': observed - standardized})
mix_standardization = pd.DataFrame(mix_rows)
mix_standardization.to_csv(error_tables / 'flat_type_mix_standardization.csv', index=False)
display(mix_standardization)
error_support['share_pct'] = 100 * error_support['n'] / error_support.groupby('group')['n'].transform('sum')
error_support['contribution_to_B4_minus_B3_MAE_SGD'] = (error_support['MAE_B4'] - error_support['MAE_B3']) * error_support['share_pct'] / 100
error_support.to_csv(error_tables / 'support_contribution.csv', index=False)
display(error_support.loc[error_support['group'].isin(['BUKIT BATOK', 'TAMPINES'])])


,town,observed_delta_MAE_SGD,delta_MAE_using_2024_flat_type_shares_SGD,composition_difference_SGD
0,BUKIT BATOK,20310.214,19555.056,755.157
1,TAMPINES,10872.986,10515.729,357.258


,group,support,n,MAE_B3,MAE_B4,signed_error_B4,share_pct,contribution_to_B4_minus_B3_MAE_SGD
5,BUKIT BATOK,unseen,87,70239.475,62695.903,62078.586,6.703,-505.617
6,BUKIT BATOK,1-9,57,31238.520,32169.502,15202.638,4.391,40.883
7,BUKIT BATOK,10-29,834,34837.364,52544.426,37964.655,64.253,11377.265
8,BUKIT BATOK,30-99,320,39739.366,80580.950,70371.549,24.653,10068.804
9,TAMPINES,unseen,53,37393.847,91019.383,90963.050,2.763,1481.832
10,TAMPINES,1-9,377,46002.084,95270.567,84507.653,19.656,9684.160
11,TAMPINES,10-29,1229,42976.088,44094.060,19291.446,64.077,716.364
12,TAMPINES,30-99,259,37616.864,47835.396,7411.627,13.504,1379.875


### 15.2 Findings and implications

**The deterioration is concentrated at the address step.** BUKIT BATOK MAE is SGD 38,932 (B0), 35,562 (B2), 38,261 (B3), and 59,242 (B4). TAMPINES is SGD 45,082, 42,910, 42,693, and 55,955. Thus categorical and functional-form preparation do not generally worsen these towns; B4 adds SGD 20,981 and SGD 13,262 MAE relative to B3. B4 is worse than B0 in all twelve 2025 months in both towns. Building-cluster bootstrap 95% intervals for B4-minus-B3 MAE are approximately SGD 15,168–26,865 and SGD 7,843–18,898, respectively.

**The errors are inside common housing groups, not just rare labels or a changed Flat-Type mix.** 4 ROOM and 5 ROOM contribute SGD 22,058 to BUKIT BATOK's net SGD 20,310 B4-minus-B0 deterioration; other groups offset part of that increase. They contribute SGD 12,652 to TAMPINES's net SGD 10,873 deterioration. Replacing 2025 Flat-Type shares by 2024 shares leaves deterioration of SGD 19,555 and SGD 10,516. The difference attributable to this limited composition standardization is only SGD 755 and SGD 357; unmeasured or within-type changes remain possible.

**Address support matters differently by town.** In TAMPINES, unseen building addresses and addresses with 1–9 historical transactions account for 430 of 1,918 test records (22.4%) but about 84.2% of the B4-minus-B3 MAE increase. Unseen buildings use the existing parent-level fallback. In BUKIT BATOK, almost all positive deterioration comes from addresses with 10–99 historical transactions, so a small-sample explanation alone is insufficient. Support is associated with error concentration, not an identified cause.

**Address encoding raises predictions and trades off magnitude against calibration.** Overall B3-to-B4 MAE improves by SGD 934, but mean signed error rises from SGD +330 to +14,603. The building-cluster 95% interval for the overall MAE change is approximately SGD -1,973 to +55, so the marginal MAE improvement is not clearly separated from zero under this exploratory uncertainty check. Mean predictions increase by SGD 26,910 in BUKIT BATOK and SGD 21,946 in TAMPINES; their B4 mean signed errors are +46,571 and +32,487.

The address coefficient is positive (0.3828). In a fixed-model sensitivity, using history only through 2023 instead of 2024 reduces mean overprediction to +1,787 overall, +33,788 in BUKIT BATOK, and +1,916 in TAMPINES. This demonstrates sensitivity to the latest historical price information. It is consistent with raw historical prices mixing location, flat composition and market-time information, but does not establish those as unique causes. BUKIT BATOK remains overpredicted even after freezing, so recent-history movement is only part of its issue.

The exact log-prediction decomposition includes the address term, intercept, smearing correction, and all changed shared coefficients; the largest reconstruction error is below 6e-15. The housing and intercept components substantially cancel, so the address coefficient alone cannot explain the full B4-minus-B3 change. The smearing factors are 1.00557 (B3) and 1.00451 (B4); their direction does not explain B4's increased mean overprediction.

**Data-practice implication.** Retain chronological encoding and report MAE, signed errors, support and subgroup results together. Pre-specify a time-normalized or residual-based historical address encoding, select it on validation data, and evaluate it on an untouched chronological holdout. Do not subtract the observed 2025 bias, switch the final policy to frozen history, or tune town-specific weights using these test results. These towns were selected after observing the original test output, so this extension remains exploratory.


### 16. Time-normalized historical address encoding
Section 15.2 recommends a pre-specified test of a time-normalized or residual-based historical address encoding. This section carries out that test. It changes no upstream cell, reuses the 2024-selected time trend, and selects between the two new encodings using **2024 validation MAE only**. The 2025 test set is used once, for final reporting.

**Diagnosis being addressed.** The upstream encoding averages *nominal* resale prices over an expanding historical window. Each additional year of history therefore mixes two different things into one feature: how expensive a building is, and how hot the market was recently. Market time is already handled by the model's own time terms, so the information enters the prediction twice and inflates 2025 forecasts. Section 15.2 shows the symptom directly: adding the nominal historical address feature shifts the signed error upward, while freezing the feature's history reduces that shift. The exact values are reported in the executed tables below.

**Two candidate encodings**, both built from one auxiliary quality-adjusted model per encoding year, fitted on strictly earlier years:

| Variant | Address feature |
|---|---|
| N0 | None (reproduces B3) |
| N1 | Nominal hierarchical price encoding (reproduces B4) |
| N2 | Hierarchical encoding of prices **deflated** to 2017-01 by a quality-adjusted index |
| N3 | Hierarchical encoding of auxiliary-model **residuals** (log scale, market time removed) |

Smoothing strengths (50 / 20 / 10), the address hierarchy, and the chronological rule are identical to the upstream encoder, so the only thing that changes is what is being averaged.

In [36]:
from pathlib import Path
 
SECTION16_OUTPUT = Path("section16_outputs")
SECTION16_OUTPUT.mkdir(exist_ok=True)
 
# The auxiliary model is the prepared specification plus full month indicators. Month indicators
# give a quality-adjusted price index; the residuals are prices net of market time and housing
# attributes. One auxiliary fit per encoding year serves both new variants.
AUX_RHS = f"{TOWN} + {TYPE} + {MODEL} + log_area + {STOREY} + lease_10 + I(lease_10 ** 2)"
AUX_FORMULA = f"log_price ~ C(month) + {AUX_RHS}"
 
 
def fit_auxiliary(sample):
    """Fit the quality-adjusted index model. Mirrors fit_hedonic without the rank assertion,
    because early reference windows can be thin; statsmodels' pinv handles that gracefully."""
    outcome, design = patsy.dmatrices(AUX_FORMULA, sample, return_type="dataframe")
    if REDUNDANT_MODEL in design.columns:
        design = design.drop(columns=REDUNDANT_MODEL)
    return sm.OLS(outcome.iloc[:, 0], design).fit()
 
 
def month_effects(fit, sample):
    """Recover the log price index, with the omitted base month set to zero."""
    effects = {
        term.split("[T.")[1][:-1]: value
        for term, value in fit.params.items()
        if term.startswith("C(month)[T.")
    }
    base = [m for m in sample["month"].unique() if m not in effects]
    assert len(base) == 1, f"Expected exactly one base month, found {base}"
    effects[base[0]] = 0.0
    return pd.Series(effects)
 
 
def hierarchical_encode(reference, values, addresses, strengths=(50, 20, 10)):
    """Generalized form of the upstream encoder: identical hierarchy, identical shrinkage,
    but any value column instead of raw resale price."""
    work = reference[address_columns].copy()
    work["_value"] = np.asarray(values)
    global_mean = work["_value"].mean()
    result = pd.DataFrame(index=addresses.index)
    parent = None
    for level, (strength, name) in enumerate(zip(strengths, ["town_te", "street_te", "address_te"]), 1):
        keys = address_columns[:level]
        stats = work.groupby(keys)["_value"].agg(["sum", "count"]).reset_index()
        if level == 1:
            stats["prior"] = global_mean
            fallback = global_mean
        else:
            stats = stats.merge(parent.rename(columns={"encoding": "prior"}), on=keys[:-1])
            fallback = result.iloc[:, -1]
        stats["encoding"] = (stats["sum"] + strength * stats["prior"]) / (stats["count"] + strength)
        parent = stats[keys + ["encoding"]]
        mapped = addresses[keys].merge(parent, on=keys, how="left", sort=False)["encoding"]
        result[name] = pd.Series(mapped.to_numpy(), index=addresses.index).fillna(fallback)
    return result
 
 
# The generalized encoder must reproduce the upstream encoding exactly on nominal prices.
_reference_2025 = hedonic.loc[hedonic["transaction_year"].lt(2025)]
_current_2025 = hedonic.loc[hedonic["transaction_year"].eq(2025)]
_replicated = hierarchical_encode(
    _reference_2025, _reference_2025["resale_price"], _current_2025[address_columns]
)["address_te"]
assert np.allclose(_replicated, address_encoding.loc[_current_2025.index, "address_te"])
print("Encoder replication check passed: generalized encoder reproduces the upstream encoding.")

Encoder replication check passed: generalized encoder reproduces the upstream encoding.


In [37]:
# Chronological construction. For encoding year Y the auxiliary model, the price index and the
# address averages all use strictly earlier years, exactly as the upstream encoding does.
deflated_parts, residual_parts, index_rows = [], [], []
 
for year in range(2018, 2026):
    reference = hedonic.loc[hedonic["transaction_year"].lt(year)]
    current = hedonic.loc[hedonic["transaction_year"].eq(year)]
    assert reference["transaction_year"].max() < year
 
    auxiliary = fit_auxiliary(reference)
    effects = month_effects(auxiliary, reference)
    deflated_price = reference["resale_price"] / np.exp(reference["month"].map(effects).to_numpy())
    residual = pd.Series(auxiliary.resid, index=reference.index).reindex(reference.index)
 
    deflated_parts.append(
        hierarchical_encode(reference, deflated_price, current[address_columns])["address_te"]
    )
    residual_parts.append(
        hierarchical_encode(reference, residual, current[address_columns])["address_te"]
    )
    index_rows.append({
        "encoding_year": year,
        "reference_last_year": int(reference["transaction_year"].max()),
        "reference_rows": len(reference),
        "encoded_rows": len(current),
        "index_latest_month": effects.index.max(),
        "index_latest_value_pct": 100 * np.expm1(effects.loc[effects.index.max()]),
        "auxiliary_R2": auxiliary.rsquared,
    })
    print(f"  encoded {year} from {len(reference):,} earlier rows", flush=True)
 
deflated_encoding = pd.concat(deflated_parts).sort_index()
residual_encoding = pd.concat(residual_parts).sort_index()
index_audit = pd.DataFrame(index_rows)
index_audit.to_csv(SECTION16_OUTPUT / "encoding_construction_audit.csv", index=False)
display(index_audit)

  encoded 2018 from 20,337 earlier rows


  encoded 2019 from 41,885 earlier rows


  encoded 2020 from 64,052 earlier rows


  encoded 2021 from 87,365 earlier rows


  encoded 2022 from 116,416 earlier rows


  encoded 2023 from 143,113 earlier rows


  encoded 2024 from 168,823 earlier rows


  encoded 2025 from 196,620 earlier rows


,encoding_year,reference_last_year,reference_rows,encoded_rows,index_latest_month,index_latest_value_pct,auxiliary_R2
0,2018,2017,20337,21548,2017-12,-1.124,0.890
1,2019,2018,41885,22167,2018-12,-1.972,0.890
2,2020,2019,64052,23313,2019-12,-1.238,0.889
3,2021,2020,87365,29051,2020-12,4.692,0.891
4,2022,2021,116416,26697,2021-12,19.264,0.898
5,2023,2022,143113,25710,2022-12,31.939,0.904
6,2024,2023,168823,27797,2023-12,38.398,0.908
7,2025,2024,196620,25009,2024-12,52.974,0.914


In [38]:
# Attach the new features to local copies. Upstream frames are untouched.
section16 = forecast.copy()
section16["log_address_deflated"] = np.log(deflated_encoding.reindex(section16.index))
section16["address_residual"] = residual_encoding.reindex(section16.index)
assert section16[["log_address_te", "log_address_deflated", "address_residual"]].notna().all().all()
 
development16 = section16.loc[section16["transaction_year"].le(2023)]
validation16 = section16.loc[section16["transaction_year"].eq(2024)]
training16 = section16.loc[section16["transaction_year"].le(2024)]
test16 = section16.loc[section16["transaction_year"].eq(2025)]
for new, old in [(development16, development), (validation16, validation),
                 (training16, training), (test16, test)]:
    assert new.index.equals(old.index)
 
# How much market-time information does each feature still carry? A feature that tracks the
# calendar is doing the time terms' job a second time.
drift_rows = []
for label, column in [("N1 nominal", "log_address_te"),
                      ("N2 deflated", "log_address_deflated"),
                      ("N3 residual", "address_residual")]:
    yearly = section16.groupby("transaction_year")[column].mean()
    standardized = (yearly - yearly.mean()) / section16[column].std()
    drift_rows.append({
        "variant": label,
        "correlation_with_time_years": section16[column].corr(section16["time_years"]),
        "2018_to_2025_drift_in_SD": standardized.loc[2025] - standardized.loc[2018],
    })
feature_drift = pd.DataFrame(drift_rows).set_index("variant")
feature_drift.to_csv(SECTION16_OUTPUT / "feature_time_drift.csv")
display(feature_drift)

,correlation_with_time_years,2018_to_2025_drift_in_SD
variant,,
N1 nominal,0.195,0.576
N2 deflated,-0.050,-0.136
N3 residual,-0.001,0.006


In [39]:
# Four variants on one shared specification. N0 and N1 must reproduce upstream B3 and B4.
base_rhs = prepared_rhs.replace("TIME_TERM", time_term)
section16_formulas = {
    "N0 no address": "log_price ~ " + base_rhs,
    "N1 nominal address": "log_price ~ " + base_rhs + " + log_address_te",
    "N2 deflated address": "log_price ~ " + base_rhs + " + log_address_deflated",
    "N3 residual address": "log_price ~ " + base_rhs + " + address_residual",
}
 
section16_predictions = pd.DataFrame(index=test16.index)
section16_fits = {}
section16_rows = []
 
for name, formula in section16_formulas.items():
    _, validation_prediction = predict_price(formula, development16, validation16, log_target=True)
    fit, test_prediction = predict_price(formula, training16, test16, log_target=True)
    section16_fits[name] = fit
    section16_predictions[name] = test_prediction
    error = test_prediction - test16["resale_price"]
    section16_rows.append({
        "variant": name,
        "validation_MAE_SGD": (validation_prediction - validation16["resale_price"]).abs().mean(),
        **price_scores(test16["resale_price"], test_prediction, test16["town"]),
        "overprediction_pct": 100 * error.gt(0).mean(),
    })
 
# price_scores names this column mean_error_SGD; Section 15 calls the same quantity
# signed_error_SGD. Rename once here so the two sections report under one name.
section16_results = pd.DataFrame(section16_rows).set_index("variant").rename(
    columns={"mean_error_SGD": "signed_error_SGD"})
 
# Reproduction checks against the upstream pipeline.
assert np.allclose(section16_predictions["N0 no address"], pipeline_predictions["B3 + log target"])
assert np.allclose(section16_predictions["N1 nominal address"],
                   pipeline_predictions["B4 + historical address"])
print("Reproduction check passed: N0 matches B3 and N1 matches B4 on the 2025 test set.")
 
section16_results.to_csv(SECTION16_OUTPUT / "variant_results.csv")
display(section16_results)

Reproduction check passed: N0 matches B3 and N1 matches B4 on the 2025 test set.


,validation_MAE_SGD,MAE_SGD,RMSE_SGD,signed_error_SGD,Town_macro_MAE_SGD,worst_Town_MAE_SGD,overprediction_pct
variant,,,,,,,
N0 no address,44125.319,47231.629,63323.880,329.726,51241.563,85006.510,50.278
N1 nominal address,42453.457,46297.934,62040.999,14603.157,48728.132,74160.857,58.823
N2 deflated address,40395.186,42260.121,56882.046,1556.575,45427.624,67451.885,50.458
N3 residual address,31543.841,31605.673,42716.804,1829.843,35042.216,61062.744,48.603


In [40]:
# Selection rule, stated and applied before reading any 2025 outcome for the new variants.
candidates = ["N2 deflated address", "N3 residual address"]
selected_variant = section16_results.loc[candidates, "validation_MAE_SGD"].idxmin()
print(f"Selected on 2024 validation MAE only: {selected_variant}")
display(section16_results.loc[candidates, ["validation_MAE_SGD"]])

Selected on 2024 validation MAE only: N3 residual address


,validation_MAE_SGD
variant,
N2 deflated address,40395.186
N3 residual address,31543.841


In [41]:
# Group breakdown on the same cleaned test records, including Section 15's two focus towns.
group_rows = []
for label, mask in [("ALL", pd.Series(True, index=test16.index))] + [
    (town, test16["town"].eq(town)) for town in FOCUS_TOWNS
]:
    part = test16.loc[mask]
    for name in section16_formulas:
        error = section16_predictions.loc[part.index, name] - part["resale_price"]
        group_rows.append({
            "group": label,
            "variant": name,
            "n": len(part),
            "MAE_SGD": error.abs().mean(),
            "signed_error_SGD": error.mean(),
            "overprediction_pct": 100 * error.gt(0).mean(),
        })
section16_groups = pd.DataFrame(group_rows)
section16_groups.to_csv(SECTION16_OUTPUT / "group_results.csv", index=False)
display(section16_groups.pivot(index="group", columns="variant", values="signed_error_SGD")
        .reindex(["ALL"] + FOCUS_TOWNS))

variant,N0 no address,N1 nominal address,N2 deflated address,N3 residual address
group,,,,
ALL,329.726,14603.157,1556.575,1829.843
BUKIT BATOK,19660.834,46570.728,24698.202,13322.317
TAMPINES,10540.931,32486.562,8879.915,2347.418


In [42]:
# Does the fix work where Section 15 found it broken? Historical-address support bands.
support_rows = []
for band, part in test16.join(audit[["support_band"]]).groupby("support_band", observed=True):
    row = {"support": str(band), "n": len(part)}
    for name in section16_formulas:
        error = section16_predictions.loc[part.index, name] - part["resale_price"]
        row[f"MAE_{name.split()[0]}"] = error.abs().mean()
        row[f"signed_{name.split()[0]}"] = error.mean()
    support_rows.append(row)
section16_support = pd.DataFrame(support_rows)
section16_support.to_csv(SECTION16_OUTPUT / "support_band_results.csv", index=False)
display(section16_support)

,support,n,MAE_N0,signed_N0,MAE_N1,signed_N1,MAE_N2,signed_N2,MAE_N3,signed_N3
0,unseen,816,66905.740,-16090.387,66405.495,-8936.127,65551.246,-31675.710,46644.501,-6003.811
1,1-9,1927,53370.703,-8301.021,57544.537,19502.450,47843.140,-5663.016,34018.706,-3790.075
2,10-29,14443,44842.867,-602.369,42948.269,14425.240,39591.173,3175.002,30981.631,-448.196
3,30-99,7674,47892.009,5767.947,47253.951,15768.407,42931.487,3324.722,30315.489,7995.580
4,100+,149,57628.025,12139.012,66182.142,37386.086,66632.699,28979.327,44976.931,20673.424


In [43]:
short = {name: name.split()[0] for name in section16_formulas}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
 
overall = section16_results.reindex(list(section16_formulas))
axes[0].plot([short[n] for n in overall.index], overall["MAE_SGD"] / 1000, marker="o", label="MAE")
axes[0].plot([short[n] for n in overall.index], overall["signed_error_SGD"] / 1000,
             marker="s", label="Signed error")
axes[0].axhline(0, color="gray", lw=.8)
axes[0].set(title="2025 test: magnitude versus calibration", ylabel="SGD thousands", xlabel="Variant")
axes[0].legend(fontsize=8)
 
pivot = section16_groups.pivot(index="group", columns="variant", values="signed_error_SGD")
pivot = pivot.reindex(["ALL"] + FOCUS_TOWNS)[list(section16_formulas)]
pivot.rename(columns=short).plot.bar(ax=axes[1], width=.8)
axes[1].axhline(0, color="gray", lw=.8)
axes[1].set(title="Mean signed error by town", ylabel="Predicted minus actual (SGD)", xlabel="")
axes[1].tick_params(axis="x", labelrotation=0)
axes[1].legend(fontsize=8)
 
yearly = section16.groupby("transaction_year")[
    ["log_address_te", "log_address_deflated", "address_residual"]
].mean()
standardized = (yearly - yearly.mean()) / section16[yearly.columns].std()
for column, label in zip(yearly.columns, ["N1 nominal", "N2 deflated", "N3 residual"]):
    axes[2].plot(standardized.index, standardized[column], marker="o", label=label)
axes[2].axhline(0, color="gray", lw=.8)
axes[2].set(title="Does the address feature track the calendar?",
            ylabel="Mean feature value (SD units)", xlabel="Transaction year")
axes[2].legend(fontsize=8)
 
fig.suptitle("Time-normalized address encoding: calibration without losing accuracy")
fig.tight_layout()
fig.savefig(SECTION16_OUTPUT / "01_time_normalized_address.png", dpi=180, bbox_inches="tight")
plt.show()

/tmp/ipykernel_61370/182320468.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
# The size of the MAE reduction is large enough that it must be shown not to come from leakage.
# Two checks: the magnitude of the fitted coefficient, and where the gain is concentrated.
residual_coefficient = section16_fits["N3 residual address"].params["address_residual"]
print(f"N3 coefficient on address_residual: {residual_coefficient:.4f}")
 
# A shrunken historical location deviation entering a log-price model should carry a coefficient
# near 1; leakage of the current year's price would push it well above 1. The gain should also be
# concentrated where historical support exists.
display(section16_support)

N3 coefficient on address_residual: 1.0281


,support,n,MAE_N0,signed_N0,MAE_N1,signed_N1,MAE_N2,signed_N2,MAE_N3,signed_N3
0,unseen,816,66905.740,-16090.387,66405.495,-8936.127,65551.246,-31675.710,46644.501,-6003.811
1,1-9,1927,53370.703,-8301.021,57544.537,19502.450,47843.140,-5663.016,34018.706,-3790.075
2,10-29,14443,44842.867,-602.369,42948.269,14425.240,39591.173,3175.002,30981.631,-448.196
3,30-99,7674,47892.009,5767.947,47253.951,15768.407,42931.487,3324.722,30315.489,7995.580
4,100+,149,57628.025,12139.012,66182.142,37386.086,66632.699,28979.327,44976.931,20673.424


In [45]:
n0 = section16_results.loc["N0 no address"]
n1 = section16_results.loc["N1 nominal address"]
n2 = section16_results.loc["N2 deflated address"]
best = section16_results.loc[selected_variant]
selected_short = selected_variant.split()[0]
groups = section16_groups.set_index(["group", "variant"])
unseen_band, largest_band = section16_support.iloc[0], section16_support.iloc[-1]
band_gain = 100 * (1 - section16_support[f"MAE_{selected_short}"] / section16_support["MAE_N0"])
 
display(Markdown(f"""
**Result.** On the same **{len(test16):,}** cleaned test records, the nominal encoding (N1) improves MAE over no address
(N0) by **SGD {n0['MAE_SGD'] - n1['MAE_SGD']:,.0f}** but moves mean signed error from
**SGD {n0['signed_error_SGD']:+,.0f}** to **SGD {n1['signed_error_SGD']:+,.0f}**. The selected
time-normalized encoding, **{selected_variant}**, reaches MAE
**SGD {best['MAE_SGD']:,.0f}** with mean signed error **SGD {best['signed_error_SGD']:+,.0f}**, and
overprediction share **{best['overprediction_pct']:.1f}%** against N1's
**{n1['overprediction_pct']:.1f}%**.
 
In the two towns Section 15 identified, mean signed error moves from
**SGD {groups.loc[('BUKIT BATOK', 'N1 nominal address'), 'signed_error_SGD']:+,.0f}** to
**SGD {groups.loc[('BUKIT BATOK', selected_variant), 'signed_error_SGD']:+,.0f}** in BUKIT BATOK and
from **SGD {groups.loc[('TAMPINES', 'N1 nominal address'), 'signed_error_SGD']:+,.0f}** to
**SGD {groups.loc[('TAMPINES', selected_variant), 'signed_error_SGD']:+,.0f}** in TAMPINES.
 
**Interpretation.** The nominal encoding's correlation with calendar time is
**{feature_drift.loc['N1 nominal', 'correlation_with_time_years']:.3f}**, against
**{feature_drift.loc['N2 deflated', 'correlation_with_time_years']:.3f}** for the deflated encoding
and **{feature_drift.loc['N3 residual', 'correlation_with_time_years']:.3f}** for the residual
encoding, and the fitted coefficient on the residual encoding is **{residual_coefficient:.3f}**.
Both facts match Section 15.2's diagnosis: a raw historical price average carries market time that
the model's time terms already represent, the duplication shows up as systematic overprediction
when the model is extrapolated to a later year, and once that component is removed the model takes
the feature at close to face value. This is a data-preparation change, not a model change: the
specification, the estimator and the training rows are identical across N1, N2 and N3.
 
Removing market time does more than correct calibration. N1 improves MAE over N0 by only
**SGD {n0['MAE_SGD'] - n1['MAE_SGD']:,.0f}**, whereas {selected_short} improves it by
**SGD {n0['MAE_SGD'] - best['MAE_SGD']:,.0f}**. The contamination was not simply an additive bias;
it was cancelling most of the feature's predictive value.
 
The support-band breakdown locates that value, and it is not mainly the individual block. MAE falls
in every band: by {band_gain.iloc[:4].min():.0f} to {band_gain.iloc[:4].max():.0f} percent in the
four bands up to 99 prior transactions, including the **{unseen_band['n']:,.0f}** test records whose
block has no prior transaction at all and whose encoding therefore falls back to the street and Town
levels, and by only {band_gain.iloc[-1]:.0f} percent in the sparse {largest_band['support']} band,
which carries the most block history and gains the least. The specification contains Town fixed
effects but no street term, so the residual encoding largely supplies street-level granularity the
model otherwise lacks. N2 makes the same point from the other side: it is marginally better
calibrated than {selected_short}
(**SGD {n2['signed_error_SGD']:+,.0f}** against **SGD {best['signed_error_SGD']:+,.0f}**) but far
less accurate (**SGD {n2['MAE_SGD']:,.0f}** against **SGD {best['MAE_SGD']:,.0f}**), because
deflating still leaves a price level that scales with floor area and flat type, quantities the
specification already contains.
 
**Boundaries.** The choice between the two new encodings used 2024 validation MAE only, and the
2025 test set was read once. These are still single-test-year results, so the ranking is not yet
shown to be stable across periods; a rolling-origin check over several test years would be the
next step. The auxiliary index is estimated from the same transactions it deflates and is therefore
subject to its own specification error. Calibration under {selected_short} rises monotonically with
historical support, from **SGD {unseen_band[f'signed_{selected_short}']:+,.0f}** in the unseen band
to **SGD {largest_band[f'signed_{selected_short}']:+,.0f}** in the {largest_band['support']} band,
so the overall figure averages over offsetting group errors. The attribution to street-level
granularity is made by elimination rather than by a direct test; adding an explicit street term to
the specification would test it head-on. Address encodings remain predictive features and are not
interpreted as causal location effects.
"""))


**Result.** On the same **25,009** cleaned test records, the nominal encoding (N1) improves MAE over no address
(N0) by **SGD 934** but moves mean signed error from
**SGD +330** to **SGD +14,603**. The selected
time-normalized encoding, **N3 residual address**, reaches MAE
**SGD 31,606** with mean signed error **SGD +1,830**, and
overprediction share **48.6%** against N1's
**58.8%**.

In the two towns Section 15 identified, mean signed error moves from
**SGD +46,571** to
**SGD +13,322** in BUKIT BATOK and
from **SGD +32,487** to
**SGD +2,347** in TAMPINES.

**Interpretation.** The nominal encoding's correlation with calendar time is
**0.195**, against
**-0.050** for the deflated encoding
and **-0.001** for the residual
encoding, and the fitted coefficient on the residual encoding is **1.028**.
Both facts match Section 15.2's diagnosis: a raw historical price average carries market time that
the model's time terms already represent, the duplication shows up as systematic overprediction
when the model is extrapolated to a later year, and once that component is removed the model takes
the feature at close to face value. This is a data-preparation change, not a model change: the
specification, the estimator and the training rows are identical across N1, N2 and N3.

Removing market time does more than correct calibration. N1 improves MAE over N0 by only
**SGD 934**, whereas N3 improves it by
**SGD 15,626**. The contamination was not simply an additive bias;
it was cancelling most of the feature's predictive value.

The support-band breakdown locates that value, and it is not mainly the individual block. MAE falls
in every band: by 30 to 37 percent in the
four bands up to 99 prior transactions, including the **816** test records whose
block has no prior transaction at all and whose encoding therefore falls back to the street and Town
levels, and by only 22 percent in the sparse 100+ band,
which carries the most block history and gains the least. The specification contains Town fixed
effects but no street term, so the residual encoding largely supplies street-level granularity the
model otherwise lacks. N2 makes the same point from the other side: it is marginally better
calibrated than N3
(**SGD +1,557** against **SGD +1,830**) but far
less accurate (**SGD 42,260** against **SGD 31,606**), because
deflating still leaves a price level that scales with floor area and flat type, quantities the
specification already contains.

**Boundaries.** The choice between the two new encodings used 2024 validation MAE only, and the
2025 test set was read once. These are still single-test-year results, so the ranking is not yet
shown to be stable across periods; a rolling-origin check over several test years would be the
next step. The auxiliary index is estimated from the same transactions it deflates and is therefore
subject to its own specification error. Calibration under N3 rises monotonically with
historical support, from **SGD -6,004** in the unseen band
to **SGD +20,673** in the 100+ band,
so the overall figure averages over offsetting group errors. The attribution to street-level
granularity is made by elimination rather than by a direct test; adding an explicit street term to
the specification would test it head-on. Address encodings remain predictive features and are not
interpreted as causal location effects.


### 16.1 Rolling-origin check across four test years
Section 16 reports a single test year. This subsection asks whether the ordering of the four variants is a property of the preparation choices or an accident of 2025. Each test year T from 2022 to 2025 is predicted by a model trained only on years before T, using the address encodings already built in Section 16, which for year T also use only years before T. Nothing is re-selected and no new encoding is constructed.

Absolute error is not comparable across years: later origins train on more data, and the market level differs. What is comparable within a year is the ranking of the four variants and the sign and size of their mean signed error.

**Categorical support across windows.** An early training window does not contain every category that appears later. The flat_model 3Gen, for example, is absent from 2018 to 2021 but present from 2022, and a regression cannot score a level it has never seen. Each test year is therefore restricted to records whose Town, Flat Type, Flat Model, storey band and calendar month all occur in that year's training window. The restriction is computed once per year and shared by all four variants, so the within-year comparison remains like for like, and the number of records removed is reported below. This is itself a finding: the composition of the resale market changes enough across a few years that out-of-time evaluation has to state which records it can score.

**Carried-over choice.** The time trend was selected once, on the 2024 validation year, by the upstream procedure in Section 13. Reusing it for the 2022 and 2023 origins means those years are predicted with a trend chosen using information from after their own test period. This affects all four variants identically, so it does not distort the comparison between them, but it does mean the earlier origins are not clean out-of-time forecasts in their own right.

In [46]:
CATEGORICAL_COLUMNS = ["town", "flat_type", "flat_model", "storey_range", "calendar_month"]
 
 
def supported_rows(train_part, test_part):
    """Test records whose categorical levels all appear in the training window."""
    mask = pd.Series(True, index=test_part.index)
    for column in CATEGORICAL_COLUMNS:
        mask &= test_part[column].isin(train_part[column].unique())
    return mask
 
 
rolling_rows, coverage_rows = [], []
for test_year in range(2022, 2026):
    train_part = section16.loc[section16["transaction_year"].lt(test_year)]
    candidate = section16.loc[section16["transaction_year"].eq(test_year)]
    assert train_part["transaction_year"].max() < test_year
 
    mask = supported_rows(train_part, candidate)
    test_part = candidate.loc[mask]
    coverage_rows.append({
        "test_year": test_year,
        "train_rows": len(train_part),
        "test_rows_available": len(candidate),
        "test_rows_dropped": int((~mask).sum()),
        "test_rows_evaluated": len(test_part),
        "dropped_pct": 100 * (~mask).sum() / len(candidate),
    })
 
    for name, formula in section16_formulas.items():
        _, prediction = predict_price(formula, train_part, test_part, log_target=True)
        error = prediction - test_part["resale_price"]
        rolling_rows.append({
            "test_year": test_year,
            "variant": name.split()[0],
            "n": len(test_part),
            "MAE_SGD": error.abs().mean(),
            "signed_error_SGD": error.mean(),
            "overprediction_pct": 100 * error.gt(0).mean(),
        })
    print(f"  origin {test_year}: trained on {len(train_part):,}, scored {len(test_part):,} "
          f"of {len(candidate):,}", flush=True)
 
rolling = pd.DataFrame(rolling_rows)
coverage = pd.DataFrame(coverage_rows).set_index("test_year")
rolling.to_csv(SECTION16_OUTPUT / "rolling_origin_results.csv", index=False)
coverage.to_csv(SECTION16_OUTPUT / "rolling_origin_coverage.csv")
 
# The 2025 origin uses the upstream training window, so it must score every test record and
# reproduce the Section 16 headline figures exactly.
assert coverage.loc[2025, "test_rows_dropped"] == 0
assert coverage.loc[2025, "test_rows_evaluated"] == len(test16)
print("Coverage check passed: the 2025 origin scores all test records.")
 
variant_order = [name.split()[0] for name in section16_formulas]
rolling_mae = rolling.pivot(index="test_year", columns="variant", values="MAE_SGD")[variant_order]
rolling_signed = rolling.pivot(index="test_year", columns="variant",
                               values="signed_error_SGD")[variant_order]
rolling_over = rolling.pivot(index="test_year", columns="variant",
                             values="overprediction_pct")[variant_order]
 
assert np.allclose(rolling_mae.loc[2025].to_numpy(),
                   section16_results.loc[list(section16_formulas), "MAE_SGD"].to_numpy())
print("Consistency check passed: the 2025 origin reproduces the Section 16 results.")
 
display(Markdown("**Evaluable records by origin**"))
display(coverage)
display(Markdown("**Mean absolute error by origin (SGD)**"))
display(rolling_mae.round(0))
display(Markdown("**Mean signed error by origin (SGD), positive means overprediction**"))
display(rolling_signed.round(0))

  origin 2022: trained on 96,079, scored 26,692 of 26,697


  origin 2023: trained on 122,776, scored 25,710 of 25,710


  origin 2024: trained on 148,486, scored 27,797 of 27,797


  origin 2025: trained on 176,283, scored 25,009 of 25,009


Coverage check passed: the 2025 origin scores all test records.
Consistency check passed: the 2025 origin reproduces the Section 16 results.


...

,train_rows,test_rows_available,test_rows_dropped,test_rows_evaluated,dropped_pct
test_year,,,,,
2022,96079,26697,5,26692,0.019
2023,122776,25710,0,25710,0.000
2024,148486,27797,0,27797,0.000
2025,176283,25009,0,25009,0.000


**Mean absolute error by origin (SGD)**

variant,N0,N1,N2,N3
test_year,,,,
2022,54445.000,46264.000,51644.000,50254.000
2023,42203.000,37112.000,38327.000,33060.000
2024,44125.000,42453.000,40395.000,31544.000
2025,47232.000,46298.000,42260.000,31606.000


**Mean signed error by origin (SGD), positive means overprediction**

variant,N0,N1,N2,N3
test_year,,,,
2022,-44768.000,-34179.000,-43891.000,-45897.000
2023,-18353.000,-4951.000,-18831.000,-20076.000
2024,-9754.000,2761.000,-10805.000,-10755.000
2025,330.000,14603.000,1557.000,1830.000


In [47]:
best_variant_by_year = rolling_mae.idxmin(axis=1)
years_won = int((best_variant_by_year == selected_short).sum())
 
stability = pd.DataFrame({
    "lowest_MAE_variant": best_variant_by_year,
    "N1_signed_error_SGD": rolling_signed["N1"].round(0),
    f"{selected_short}_signed_error_SGD": rolling_signed[selected_short].round(0),
    "N1_overprediction_pct": rolling_over["N1"].round(1),
    f"{selected_short}_overprediction_pct": rolling_over[selected_short].round(1),
})
stability.to_csv(SECTION16_OUTPUT / "rolling_origin_stability.csv")
display(stability)

,lowest_MAE_variant,N1_signed_error_SGD,N3_signed_error_SGD,N1_overprediction_pct,N3_overprediction_pct
test_year,,,,,
2022,N1,-34179.000,-45897.000,21.400,9.400
2023,N3,-4951.000,-20076.000,43.500,24.600
2024,N3,2761.000,-10755.000,51.600,36.700
2025,N3,14603.000,1830.000,58.800,48.600


In [48]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
 
for variant in variant_order:
    axes[0].plot(rolling_mae.index, rolling_mae[variant] / 1000, marker="o", label=variant)
axes[0].set(title="Mean absolute error by origin", xlabel="Test year", ylabel="SGD thousands")
axes[0].set_xticks(rolling_mae.index)
axes[0].legend(fontsize=8)
 
for variant in variant_order:
    axes[1].plot(rolling_signed.index, rolling_signed[variant] / 1000, marker="o", label=variant)
axes[1].axhline(0, color="gray", lw=.8)
axes[1].set(title="Mean signed error by origin", xlabel="Test year",
            ylabel="Predicted minus actual (SGD thousands)")
axes[1].set_xticks(rolling_signed.index)
axes[1].legend(fontsize=8)
 
fig.suptitle("Rolling-origin check: is the Section 16 ordering stable across test years?")
fig.tight_layout()
fig.savefig(SECTION16_OUTPUT / "02_rolling_origin.png", dpi=180, bbox_inches="tight")
plt.show()

/tmp/ipykernel_61370/3453335246.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [49]:
bias_by_variant = rolling_signed.abs().mean()
time_free = rolling_signed[["N0", "N2", selected_short]]
time_free_spread = (time_free.max(axis=1) - time_free.min(axis=1)).max()
nominal_gap = (rolling_signed["N1"] - rolling_signed["N0"])
exception_year = int(rolling_mae.idxmin(axis=1).ne(selected_short).idxmax()) \
    if (rolling_mae.idxmin(axis=1) != selected_short).any() else None

display(Markdown(f"""
**Accuracy is stable; calibration is not.** Across the four origins from {rolling_mae.index.min()}
to {rolling_mae.index.max()}, {selected_short} attains the lowest MAE in
**{years_won} of {len(rolling_mae)}** years, the exception being **{exception_year}**. Mean absolute
signed error tells the opposite story: **SGD {bias_by_variant['N1']:,.0f}** under the nominal
encoding against **SGD {bias_by_variant[selected_short]:,.0f}** under {selected_short}. The
calibration advantage reported in Section 16 is therefore specific to the 2025 origin and does not
generalise.

**The dominant bias belongs to the specification, not to the address feature.** In
{rolling_mae.index.min()} to {rolling_mae.index.max() - 1} every variant underpredicts, including
N0, which carries no address feature at all and whose mean signed error reaches
**SGD {rolling_signed.loc[rolling_signed.index.min(), 'N0']:+,.0f}**. The extrapolated time trend,
not the encoding, is what fails in a rapidly rising market. The three variants whose address
feature carries no market time (N0, N2 and {selected_short}) differ from one another by at most
**SGD {time_free_spread:,.0f}** in any origin, while the nominal encoding sits
**SGD {nominal_gap.min():,.0f}** to **SGD {nominal_gap.max():,.0f}** above the no-address baseline
in every one.

**This strengthens rather than weakens the Section 16 mechanism.** Time normalisation did exactly
what it was meant to do: N2 and {selected_short} now track the no-address baseline's bias, meaning
their address feature injects no time-varying component of its own. What N1 gains from
{rolling_mae.index.min()} to {rolling_mae.index.max() - 1} is not accuracy of calibration but a
cancellation of two opposed errors, an upward feature contamination against a downward trend
extrapolation. The cancellation is favourable while the market is rising and reverses in
{rolling_mae.index.max()}, which is the behaviour Section 15.2 observed. A variant whose apparent
calibration depends on two errors offsetting is not a safer choice than one whose residual bias is
inherited transparently from the specification.

**Coverage.** The earliest origin can score
**{coverage.loc[coverage.index.min(), 'test_rows_evaluated']:,.0f}** of
**{coverage.loc[coverage.index.min(), 'test_rows_available']:,.0f}** test records
(**{coverage.loc[coverage.index.min(), 'dropped_pct']:.2f}%** removed for categorical levels absent
from its training window); later origins score all of theirs. Dropped records are excluded for all
four variants alike.

**What this does and does not establish.** The comparison now rests on four test periods rather than
one. It does not make the earlier origins clean out-of-time forecasts, because the time trend was
selected once on 2024 and reused throughout; that choice is shared by all four variants, so it
affects their common level rather than their ordering, but it is plausibly part of why the common
level is so badly biased in the early origins. Training size grows with the origin year, so MAE is
expected to improve across origins for reasons unrelated to the encoding. The
{rolling_mae.index.max()} row reproduces the Section 16 figures exactly, which is asserted above.
"""))


**Accuracy is stable; calibration is not.** Across the four origins from 2022
to 2025, N3 attains the lowest MAE in
**3 of 4** years, the exception being **2022**. Mean absolute
signed error tells the opposite story: **SGD 14,123** under the nominal
encoding against **SGD 19,639** under N3. The
calibration advantage reported in Section 16 is therefore specific to the 2025 origin and does not
generalise.

**The dominant bias belongs to the specification, not to the address feature.** In
2022 to 2024 every variant underpredicts, including
N0, which carries no address feature at all and whose mean signed error reaches
**SGD -44,768**. The extrapolated time trend,
not the encoding, is what fails in a rapidly rising market. The three variants whose address
feature carries no market time (N0, N2 and N3) differ from one another by at most
**SGD 2,006** in any origin, while the nominal encoding sits
**SGD 10,589** to **SGD 14,273** above the no-address baseline
in every one.

**This strengthens rather than weakens the Section 16 mechanism.** Time normalisation did exactly
what it was meant to do: N2 and N3 now track the no-address baseline's bias, meaning
their address feature injects no time-varying component of its own. What N1 gains from
2022 to 2024 is not accuracy of calibration but a
cancellation of two opposed errors, an upward feature contamination against a downward trend
extrapolation. The cancellation is favourable while the market is rising and reverses in
2025, which is the behaviour Section 15.2 observed. A variant whose apparent
calibration depends on two errors offsetting is not a safer choice than one whose residual bias is
inherited transparently from the specification.

**Coverage.** The earliest origin can score
**26,692** of
**26,697** test records
(**0.02%** removed for categorical levels absent
from its training window); later origins score all of theirs. Dropped records are excluded for all
four variants alike.

**What this does and does not establish.** The comparison now rests on four test periods rather than
one. It does not make the earlier origins clean out-of-time forecasts, because the time trend was
selected once on 2024 and reused throughout; that choice is shared by all four variants, so it
affects their common level rather than their ordering, but it is plausibly part of why the common
level is so badly biased in the early origins. Training size grows with the origin year, so MAE is
expected to improve across origins for reasons unrelated to the encoding. The
2025 row reproduces the Section 16 figures exactly, which is asserted above.


### Conclusion
Section 16 evaluated the encoding on one test year and Section 16.1 repeated it on four. The two agree on the mechanism and disagree on one consequence.

Time normalisation works. The address feature's correlation with calendar time falls from 0.195 to -0.001, and the signed errors of N0, N2 and N3 then track one another within SGD 2,006 in every origin. The accuracy gain is large and durable: MAE on the 2025 origin falls from SGD 47,232 with no address feature to SGD 31,606, and the residual encoding is best in three of four origins. Most of that gain is street-level rather than block-level, since error falls by 30 to 37 percent even for the 816 records whose block has no prior transaction, and by only 22 percent in the band with the most block history.

The calibration result does not generalise. On 2025 the residual encoding cuts mean signed error from SGD +14,603 to SGD +1,830, but across the four origins the nominal encoding is the less biased, SGD 14,123 against SGD 19,639. The reason is that the dominant bias is not the encoding's: from 2022 to 2024 every variant underpredicts, including N0, which has no address feature and reaches SGD -44,768 in 2022. The extrapolated time trend fails in a rising market, and the nominal encoding looks better calibrated in those years only because its upward contamination offsets that failure, sitting SGD 10,589 to SGD 14,273 above the no-address baseline throughout.

Two things follow. The claim that historical address encoding introduces overprediction describes the 2025 origin, not the encoding in general, and should be stated with its period. The residual encoding should be preferred for accuracy rather than presented as a general fix for bias, because the remaining bias belongs mainly to the time specification and should be addressed there.


### 17. Governance privacy
Earlier sections treat the published fields as given and ask what can be modelled from them. This section asks the prior question: what was decided before publication, and what does that decision cost and buy. HDB does not publish a unit number, an exact floor, or a transaction date. It publishes a block, a three-floor band, and a month. Those are disclosure controls, and they are the reason several limitations recorded elsewhere in this notebook exist at all.

Two questions are answered with the same cleaned record frame used throughout the analysis:

1. **How identifying is the released combination?** A record is *unique* when no other transaction shares its quasi-identifiers. A unique record attaches one resale price to one household, recoverable by anyone who knows where that household lived and roughly when they sold.
2. **What do the coarsening choices buy?** Spatial and temporal precision are varied over a grid, holding everything else fixed, so the privacy value of each released level can be read directly rather than asserted.

Using the cleaned frame keeps the duplicate and lease-consistency decisions uniform across the notebook. It changes the evaluated record population, but not the fields whose disclosure precision is being compared.


In [50]:
SECTION17_OUTPUT = Path("section17_outputs")
SECTION17_OUTPUT.mkdir(exist_ok=True)
 
published = df.copy()
assert not published.duplicated(subset=list(raw.columns)).any()
assert published.index.intersection(lease_excluded.index).empty
assert published.index.intersection(duplicate_excluded.index).empty
published["date_year"] = published["month"].str.slice(0, 4)
published["date_quarter"] = pd.PeriodIndex(published["month"], freq="M").asfreq("Q").astype(str)
 
 
def k_profile(frame, keys, label):
    """Group-size profile for one quasi-identifier set. k is the number of records sharing a
    combination; k = 1 means the record is unique and its price is attributable to one household."""
    group_size = frame.groupby(keys, sort=False)[keys[0]].transform("size")
    assert group_size.sum() == sum(
        n * n for n in frame.groupby(keys, sort=False).size()
    ), "group sizes are inconsistent"
    return {
        "quasi_identifiers": label,
        "fields": len(keys),
        "groups": int(frame.groupby(keys, sort=False).ngroups),
        "unique_pct": 100 * float((group_size == 1).mean()),
        "k_at_most_2_pct": 100 * float((group_size <= 2).mean()),
        "k_at_most_5_pct": 100 * float((group_size <= 5).mean()),
        "median_k": float(group_size.median()),
    }
 
 
progressive_sets = [
    (["town"], "Town"),
    (["town", "street_name"], "+ Street"),
    (["town", "street_name", "block"], "+ Block"),
    (["town", "street_name", "block", "flat_type"], "+ Flat Type"),
    (["town", "street_name", "block", "flat_type", "storey_range"], "+ Storey band"),
    (["town", "street_name", "block", "flat_type", "storey_range", "month"], "+ Month (as released)"),
    (["town", "street_name", "block", "flat_type", "storey_range", "month", "floor_area_sqm"],
     "+ Floor area"),
]
 
disclosure = pd.DataFrame([k_profile(published, keys, label) for keys, label in progressive_sets])
disclosure.to_csv(SECTION17_OUTPUT / "disclosure_profile.csv", index=False)
display(disclosure.round(2))

,quasi_identifiers,fields,groups,unique_pct,k_at_most_2_pct,k_at_most_5_pct,median_k
0,Town,1,26,0.000,0.000,0.000,11198.000
1,+ Street,2,576,0.000,0.000,0.000,615.000
2,+ Block,3,9670,0.040,0.120,0.910,27.000
3,+ Flat Type,4,16715,0.380,1.080,5.460,18.000
4,+ Storey band,5,69328,8.270,22.520,65.500,4.000
5,+ Month (as released),6,216136,95.130,99.750,100.000,1.000
6,+ Floor area,7,217375,96.220,99.810,100.000,1.000


In [51]:
# What the coarsening decisions buy. Spatial and temporal precision are varied over a grid while
# Flat Type and the storey band are held at their released precision, so each cell is the share of
# unique records under one hypothetical release policy.
date_levels = [("Year", "date_year"), ("Quarter", "date_quarter"), ("Month (as released)", "month")]
space_levels = [
    ("Town", ["town"]),
    ("Town + Street", ["town", "street_name"]),
    ("Town + Street + Block (as released)", ["town", "street_name", "block"]),
]
held_fixed = ["flat_type", "storey_range"]
 
grid = pd.DataFrame(
    [[k_profile(published, space_keys + held_fixed + [date_column], "")["unique_pct"]
      for _, space_keys in space_levels] for _, date_column in date_levels],
    index=[name for name, _ in date_levels],
    columns=[name for name, _ in space_levels],
)
grid.index.name = "Date precision"
grid.columns.name = "Location precision"
grid.to_csv(SECTION17_OUTPUT / "precision_grid.csv")
display(grid.round(2))
 
released_unique = grid.loc["Month (as released)", "Town + Street + Block (as released)"]
coarsest_unique = grid.loc["Year", "Town"]

Location precision,Town,Town + Street,Town + Street + Block (as released)
Date precision,,,
Year,0.450,8.820,62.210
Quarter,2.780,32.960,86.890
Month (as released),9.110,60.110,95.130


In [52]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
 
bars = axes[0].barh(disclosure["quasi_identifiers"], disclosure["unique_pct"], color="#d1495b")
axes[0].bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8, color="#333333")
axes[0].invert_yaxis()
axes[0].set(title="Share of records that are unique, as fields accumulate",
            xlabel="Unique records (%)")
axes[0].set_xlim(0, max(disclosure["unique_pct"]) * 1.18)
axes[0].grid(axis="x", alpha=.3)
axes[0].set_axisbelow(True)
 
sns.heatmap(grid, annot=True, fmt=".1f", cmap="Reds", vmin=0, cbar_kws={"label": "Unique (%)"},
            linewidths=2, linecolor="white", annot_kws={"fontsize": 9}, ax=axes[1])
axes[1].set(title="Unique records under alternative release precisions (%)",
            xlabel="Location precision", ylabel="Date precision")
axes[1].tick_params(axis="x", labelrotation=12)
 
fig.suptitle("Disclosure risk is created by precision, not by volume")
fig.tight_layout()
fig.savefig(SECTION17_OUTPUT / "01_disclosure_risk.png", dpi=180, bbox_inches="tight")
plt.show()

/tmp/ipykernel_61370/1133383742.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [53]:
# The governance record this notebook can assert, as opposed to assume.
governance = pd.DataFrame([
    ("Source", f"HDB resale flat prices, data.gov.sg dataset {metadata['dataset_id']}"),
    ("Source URL", metadata["source_url"]),
    ("Licence", "Singapore Open Data Licence v1.0 (https://data.gov.sg/open-data-licence); "
                "attribution required, no warranty of accuracy, publisher may revise"),
    ("Retrieved at", metadata["retrieved_at"]),
    ("Snapshot window", f"{metadata['snapshot_first_month']} to {metadata['snapshot_last_month']}"),
    ("Snapshot rows", f"{metadata['snapshot_rows']:,}"),
    ("Integrity control", f"SHA-256 {metadata['snapshot_sha256'][:16]}..., asserted in Section 2 "
                          f"on every run"),
    ("Refresh policy", "REFRESH_DATA is False by default. The publisher revises historical records, "
                       "so a refresh changes the test period and invalidates the fixed figures "
                       "quoted in Sections 15 to 16; a refresh therefore requires re-running and "
                       "re-reading those sections, not only re-executing them."),
    ("Retention", "The raw snapshot and per-record processed outputs are kept out of version "
                  "control; only aggregate tables and figures are committed."),
    ("Personal data", "No direct identifiers are present. Quasi-identifiers are released at reduced "
                      "precision by the publisher, and the residual uniqueness is quantified above."),
    ("Permitted use here", "Aggregate description and prediction. Records are not linked to "
                           "individuals, addresses are not geocoded to units, and no attempt is "
                           "made to reconstruct withheld precision."),
], columns=["Field", "Value"]).set_index("Field")
 
pd.set_option("display.max_colwidth", None)
display(governance)
governance.to_csv(SECTION17_OUTPUT / "governance_record.csv")

,Value
Field,
Source,"HDB resale flat prices, data.gov.sg dataset d_8b84c4ee58e3cfc0ece0d773c8ca6abc"
Source URL,https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view
Licence,"Singapore Open Data Licence v1.0 (https://data.gov.sg/open-data-licence); attribution required, no warranty of accuracy, publisher may revise"
Retrieved at,2026-10-11T21:23:29.895039+08:00
Snapshot window,2017-01 to 2025-12
Snapshot rows,"222,066"
Integrity control,"SHA-256 af2ed6574d542771..., asserted in Section 2 on every run"
Refresh policy,"REFRESH_DATA is False by default. The publisher revises historical records, so a refresh changes the test period and invalidates the fixed figures quoted in Sections 15 to 16; a refresh therefore requires re-running and re-reading those sections, not only re-executing them."
Retention,The raw snapshot and per-record processed outputs are kept out of version control; only aggregate tables and figures are committed.


In [54]:
released_row = disclosure.loc[disclosure["quasi_identifiers"].eq("+ Month (as released)")].iloc[0]
with_area = disclosure.iloc[-1]
block_only = disclosure.loc[disclosure["quasi_identifiers"].eq("+ Block")].iloc[0]
storey_row = disclosure.loc[disclosure["quasi_identifiers"].eq("+ Storey band")].iloc[0]
street_level = grid.loc["Month (as released)", "Town + Street"]
year_at_block = grid.loc["Year", "Town + Street + Block (as released)"]
town_at_month = grid.loc["Month (as released)", "Town"]
 
display(Markdown(f"""
### Conclusion
 
At the precision actually released, **{released_row['unique_pct']:.1f}%** of the
{len(published):,} transactions are unique on Town, street, block, Flat Type, storey band
and month, and **{released_row['k_at_most_5_pct']:.2f}%** sit in groups of five or fewer. The
dataset offers no k-anonymity at the record level. Location alone does not cause this: Town, street
and block together leave only **{block_only['unique_pct']:.2f}%** of records unique, because many
flats share a block. Uniqueness appears when the storey band is added, raising it to
**{storey_row['unique_pct']:.1f}%**, and then when the month is added, raising it to
**{released_row['unique_pct']:.1f}%**. It reaches **{with_area['unique_pct']:.1f}%** for an
adversary who also knows the flat's floor area, which a neighbour, agent or former owner would. A
unique record attaches one sale price to one household for anyone with ordinary knowledge of where
that household lived and roughly when they moved.

The grid shows that the two kinds of precision are not equally costly. Coarsening the date from
month to year, at the released location precision, lowers uniqueness from
**{released_unique:.1f}%** to **{year_at_block:.1f}%**; coarsening the location from block to Town,
at the released date precision, lowers it to **{town_at_month:.1f}%**. Location is the binding
constraint, and at block precision even an annual release would leave most records unique. Dropping
the block alone, keeping Town and street, would take uniqueness to **{street_level:.1f}%**.

That figure is worth holding against Section 16, which found the predictive value of the address
encoding to be largely street-level: error fell by 30 to 37 percent even for the 816 test records
whose block had no prior transaction, and by only 22 percent in the band with the most block
history. The field that contributes most to identifiability is therefore not the field that
contributes most to prediction. Releasing at street precision would cut uniqueness by a third
while, on this evidence, costing comparatively little accuracy. This is a hypothesis supported by
Section 16 rather than a direct test; re-estimating the encoding with the block level removed would
settle it.
"""))


### Conclusion

At the precision actually released, **95.1%** of the
221,629 transactions are unique on Town, street, block, Flat Type, storey band
and month, and **100.00%** sit in groups of five or fewer. The
dataset offers no k-anonymity at the record level. Location alone does not cause this: Town, street
and block together leave only **0.04%** of records unique, because many
flats share a block. Uniqueness appears when the storey band is added, raising it to
**8.3%**, and then when the month is added, raising it to
**95.1%**. It reaches **96.2%** for an
adversary who also knows the flat's floor area, which a neighbour, agent or former owner would. A
unique record attaches one sale price to one household for anyone with ordinary knowledge of where
that household lived and roughly when they moved.

The grid shows that the two kinds of precision are not equally costly. Coarsening the date from
month to year, at the released location precision, lowers uniqueness from
**95.1%** to **62.2%**; coarsening the location from block to Town,
at the released date precision, lowers it to **9.1%**. Location is the binding
constraint, and at block precision even an annual release would leave most records unique. Dropping
the block alone, keeping Town and street, would take uniqueness to **60.1%**.

That figure is worth holding against Section 16, which found the predictive value of the address
encoding to be largely street-level: error fell by 30 to 37 percent even for the 816 test records
whose block had no prior transaction, and by only 22 percent in the band with the most block
history. The field that contributes most to identifiability is therefore not the field that
contributes most to prediction. Releasing at street precision would cut uniqueness by a third
while, on this evidence, costing comparatively little accuracy. This is a hypothesis supported by
Section 16 rather than a direct test; re-estimating the encoding with the block level removed would
settle it.


### 17.1 The cost of block-level precision
Section 17 observes that the block is the field that creates most of the disclosure risk, and Section 16 finds that most of the encoding's predictive value is already present at the street level. Together those suggest that a release at street precision would cost little accuracy, but neither section tests it. This subsection does.

The residual encoding is rebuilt with the hierarchy truncated to two levels, Town and street, and nothing else changed: the same auxiliary models, the same smoothing strengths for the levels that remain (50 and 20), the same chronological rule, the same specification, the same training and test rows. The only difference is that the block level is never formed, so the feature cannot carry information below the street.

The result is read as a privacy-utility frontier. Each release precision implies both a share of unique records, computed in Section 17, and a best achievable test error, computed here, so the cost of each disclosure control can be stated in SGD rather than argued.

In [55]:
# Rebuild the residual encoding with the block level removed. hierarchical_encode forms one level
# per smoothing strength, so passing two strengths stops the hierarchy at Town and street.
street_parts = []
for year in range(2018, 2026):
    reference = hedonic.loc[hedonic["transaction_year"].lt(year)]
    current = hedonic.loc[hedonic["transaction_year"].eq(year)]
    assert reference["transaction_year"].max() < year
    auxiliary = fit_auxiliary(reference)
    residual = pd.Series(auxiliary.resid, index=reference.index)
    street_parts.append(
        hierarchical_encode(reference, residual, current[address_columns],
                            strengths=(50, 20))["street_te"]
    )
    print(f"  street-level encoding for {year} from {len(reference):,} earlier rows", flush=True)
 
street_encoding = pd.concat(street_parts).sort_index()
 
section17_frame = section16.copy()
section17_frame["address_residual_street"] = street_encoding.reindex(section17_frame.index)
assert section17_frame["address_residual_street"].notna().all()
 
training17 = section17_frame.loc[section17_frame["transaction_year"].le(2024)]
test17 = section17_frame.loc[section17_frame["transaction_year"].eq(2025)]
assert test17.index.equals(test16.index)
 
street_formula = "log_price ~ " + base_rhs + " + address_residual_street"
street_fit, street_prediction = predict_price(street_formula, training17, test17, log_target=True)
street_error = street_prediction - test17["resale_price"]
print(f"S3 coefficient on address_residual_street: "
      f"{street_fit.params['address_residual_street']:.4f}")

  street-level encoding for 2018 from 20,337 earlier rows


  street-level encoding for 2019 from 41,885 earlier rows


  street-level encoding for 2020 from 64,052 earlier rows


  street-level encoding for 2021 from 87,365 earlier rows


  street-level encoding for 2022 from 116,416 earlier rows


  street-level encoding for 2023 from 143,113 earlier rows


  street-level encoding for 2024 from 168,823 earlier rows


  street-level encoding for 2025 from 196,620 earlier rows


S3 coefficient on address_residual_street: 1.0402


In [56]:
n0 = section16_results.loc["N0 no address"]
n3 = section16_results.loc[selected_variant]
 
frontier = pd.DataFrame([
    {
        "release_precision": "Town",
        "address_feature": "none (N0)",
        "unique_pct": grid.loc["Month (as released)", "Town"],
        "MAE_SGD": n0["MAE_SGD"],
        "signed_error_SGD": n0["signed_error_SGD"],
    },
    {
        "release_precision": "Town + Street",
        "address_feature": "street residual (S3)",
        "unique_pct": grid.loc["Month (as released)", "Town + Street"],
        "MAE_SGD": street_error.abs().mean(),
        "signed_error_SGD": street_error.mean(),
    },
    {
        "release_precision": "Town + Street + Block (as released)",
        "address_feature": f"block residual ({selected_short})",
        "unique_pct": grid.loc["Month (as released)", "Town + Street + Block (as released)"],
        "MAE_SGD": n3["MAE_SGD"],
        "signed_error_SGD": n3["signed_error_SGD"],
    },
]).set_index("release_precision")
 
total_gain = n0["MAE_SGD"] - n3["MAE_SGD"]
street_gain = n0["MAE_SGD"] - frontier.loc["Town + Street", "MAE_SGD"]
retained_pct = 100 * street_gain / total_gain
block_marginal = frontier.loc["Town + Street", "MAE_SGD"] - n3["MAE_SGD"]
uniqueness_cost = (grid.loc["Month (as released)", "Town + Street + Block (as released)"]
                   - grid.loc["Month (as released)", "Town + Street"])
 
frontier["MAE_gain_vs_Town_SGD"] = n0["MAE_SGD"] - frontier["MAE_SGD"]
frontier.to_csv(SECTION17_OUTPUT / "privacy_utility_frontier.csv")
display(frontier.round(2))
 
# The claims in the conclusion are directional, so they are asserted rather than assumed.
assert 0 <= retained_pct <= 100, "street level did not land between the Town and block variants"
print(f"Street level retains {retained_pct:.1f}% of the block-level accuracy gain.")

,address_feature,unique_pct,MAE_SGD,signed_error_SGD,MAE_gain_vs_Town_SGD
release_precision,,,,,
Town,none (N0),9.110,47231.630,329.730,0.000
Town + Street,street residual (S3),60.110,35594.740,1517.870,11636.880
Town + Street + Block (as released),block residual (N3),95.130,31605.670,1829.840,15625.960


Street level retains 74.5% of the block-level accuracy gain.


In [57]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(frontier["unique_pct"], frontier["MAE_SGD"] / 1000, marker="o", markersize=9,
        linewidth=2, color="#345995")
for label, row in frontier.iterrows():
    ax.annotate(f"{label}\n{row['address_feature']}",
                (row["unique_pct"], row["MAE_SGD"] / 1000),
                textcoords="offset points", xytext=(10, 10), fontsize=8.5, color="#333333")
ax.set(title="Privacy-utility frontier for location precision",
       xlabel="Records unique in the released data (%)",
       ylabel="2025 test MAE (SGD thousands)")
ax.set_xlim(0, 108)
ax.grid(alpha=.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(SECTION17_OUTPUT / "02_privacy_utility_frontier.png", dpi=180, bbox_inches="tight")
plt.show()

/tmp/ipykernel_61370/513236133.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [58]:
town_to_street = frontier.loc["Town + Street", "unique_pct"] - frontier.loc["Town", "unique_pct"]
efficiency_street = street_gain / town_to_street
efficiency_block = block_marginal / uniqueness_cost
assert efficiency_block < efficiency_street

display(Markdown(f"""
### Conclusion
 
Truncating the address hierarchy at the street level gives a 2025 test MAE of
**SGD {frontier.loc['Town + Street', 'MAE_SGD']:,.0f}**, against
**SGD {n0['MAE_SGD']:,.0f}** with no address feature and **SGD {n3['MAE_SGD']:,.0f}** with the full
hierarchy. The street level therefore recovers **{retained_pct:.1f}%** of the accuracy that the
address encoding provides, and the block level adds the remaining
**SGD {block_marginal:,.0f}**. On the privacy side that block level raises the share of unique
records by **{uniqueness_cost:.1f}** percentage points, from
**{grid.loc['Month (as released)', 'Town + Street']:.1f}%** to
**{grid.loc['Month (as released)', 'Town + Street + Block (as released)']:.1f}%**.
 
This converts the hypothesis at the end of Section 17.1 into a measured trade-off, and the two
steps are not equally efficient. Moving from Town to street precision buys
**SGD {street_gain:,.0f}** of accuracy for **{town_to_street:.1f}** percentage points of
uniqueness, or **SGD {efficiency_street:,.0f}** per point. The further move from street to block
buys **SGD {block_marginal:,.0f}** for **{uniqueness_cost:.1f}** points, or
**SGD {efficiency_block:,.0f}** per point: about half as much accuracy for each point of privacy
given up. A publisher willing to accept **SGD {block_marginal:,.0f}** of additional error on a mean
transaction price of about SGD 653,000 could withhold the block and leave
**{100 - frontier.loc['Town + Street', 'unique_pct']:.1f}%** of records non-unique rather than
**{100 - frontier.loc['Town + Street + Block (as released)', 'unique_pct']:.1f}%**.
 
**Boundaries.** This is one test on one origin, inheriting the specification, the 2024-selected time
trend and the smoothing strengths from the sections above, and the street-level variant was not
re-tuned after truncation; a shrinkage strength chosen for a two-level hierarchy might do slightly
better. The frontier describes what this specification can extract at each precision, not an upper
bound on what any method could. The uniqueness figures assume an adversary who knows Flat Type and
the storey band, which Section 17 states explicitly. No withheld field is reconstructed anywhere in
this test; the comparison only removes information.
"""))


### Conclusion

Truncating the address hierarchy at the street level gives a 2025 test MAE of
**SGD 35,595**, against
**SGD 47,232** with no address feature and **SGD 31,606** with the full
hierarchy. The street level therefore recovers **74.5%** of the accuracy that the
address encoding provides, and the block level adds the remaining
**SGD 3,989**. On the privacy side that block level raises the share of unique
records by **35.0** percentage points, from
**60.1%** to
**95.1%**.

This converts the hypothesis at the end of Section 17.1 into a measured trade-off, and the two
steps are not equally efficient. Moving from Town to street precision buys
**SGD 11,637** of accuracy for **51.0** percentage points of
uniqueness, or **SGD 228** per point. The further move from street to block
buys **SGD 3,989** for **35.0** points, or
**SGD 114** per point: about half as much accuracy for each point of privacy
given up. A publisher willing to accept **SGD 3,989** of additional error on a mean
transaction price of about SGD 653,000 could withhold the block and leave
**39.9%** of records non-unique rather than
**4.9%**.

**Boundaries.** This is one test on one origin, inheriting the specification, the 2024-selected time
trend and the smoothing strengths from the sections above, and the street-level variant was not
re-tuned after truncation; a shrinkage strength chosen for a two-level hierarchy might do slightly
better. The frontier describes what this specification can extract at each precision, not an upper
bound on what any method could. The uniqueness figures assume an adversary who knows Flat Type and
the storey band, which Section 17 states explicitly. No withheld field is reconstructed anywhere in
this test; the comparison only removes information.


### Team contribution and verification record


| Team member | Contribution |
|---|---|
| Gu Shucheng | Core research framing, data quality, EDA, hedonic sequence, imbalance analysis, and preparation comparison |
| Wu Yuyao | Common-support restrictions and control-order sensitivity |
| Zhong ShangRui | Town-level prediction-error diagnosis and address-encoding mechanism |
| Li Zijing | Time-normalized address encoding, rolling-origin evaluation, and privacy–utility analysis |

